In [1]:
import pandas as pd
import re

DATASET_PATH = "/kaggle/input/datasets/shruteesaha/51k-after-punctuation-injection/51k_after_punctuation_injection.csv"
KARAK_RULES_PATH = "/kaggle/input/datasets/shruteesaha/karak-rule/case_karak error rule (2).xlsx"

df = pd.read_csv(DATASET_PATH)

rules_df = pd.read_excel(KARAK_RULES_PATH)

correct_words = (
    rules_df["Correct Word"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

results = []

for word in correct_words:

    pattern = r'(?<!\S)' + re.escape(word) + r'(?!\S)'

    freq = df["sentence"].astype(str).str.count(pattern).sum()

    results.append({
        "Correct Word": word,
        "Frequency": int(freq)
    })

freq_df = pd.DataFrame(results)

freq_df = freq_df.sort_values(
    by="Frequency",
    ascending=False
)

print(freq_df.to_string(index=False))

freq_df.to_csv(
    "/kaggle/working/karak_word_frequencies.csv",
    index=False
)

print("\nSaved: /kaggle/working/karak_word_frequencies.csv")

       Correct Word  Frequency
               দেশত        163
                ঘৰত        148
               দিনত        141
                মনত        121
              জিলাত        104
            কিতাপখন         83
              গাঁৱত         79
            শ্ৰেণীত         70
              মঞ্চত         48
              বজাৰত         46
             হাতেৰে         42
               নদীত         38
               ৰজাক         36
     বিদ্যালয়ৰ পৰা         34
             কুমাৰে         31
              পথাৰত         29
              ৰোগীক         27
              অতীতত         25
            ঘৰৰ পৰা         24
             মহিলাক         21
             পানীৰে         21
              দেশলৈ         20
           উপত্যকাত         19
              আকাশত         19
        বিদ্যালয়লৈ         17
             চোতালত         17
             পুত্ৰক         15
              চকুৰে         15
            শিক্ষকে         15
             নদীয়ে         15
              কামটো         15
        

In [2]:
print(freq_df["Frequency"].sum())                 # all rules
print(freq_df[freq_df["Frequency"] > 0]["Frequency"].sum())  # only usable rules

2133
2133


In [3]:
import pandas as pd
import random
import re
from collections import defaultdict

# =========================
# PATHS
# =========================
INPUT_PATH = "/kaggle/input/datasets/shruteesaha/51k-after-punctuation-injection/51k_after_punctuation_injection.csv"

KARAK_RULES_PATH = "/kaggle/input/datasets/shruteesaha/karak-rule/case_karak error rule (2).xlsx"

OUTPUT_PATH = "/kaggle/working/51k_after_tense_pronoun_pratyaya_number_person_verb_punctuation_karak_injection.csv"

REPORT_PATH = "/kaggle/working/karak_injection_report.csv"

# =========================
# SETTINGS
# =========================
RANDOM_SEED = 44
KARAK_LABEL = "7"   # change if your Karak label number is different

# Karak frequency: 14 / 343
KARAK_RATIO = 14 / 343

random.seed(RANDOM_SEED)

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(INPUT_PATH)
rules_df = pd.read_excel(KARAK_RULES_PATH)

df["sentence"] = df["sentence"].astype(str)

if "error_label" not in df.columns:
    df["error_label"] = "0"

df["error_label"] = df["error_label"].astype(str)

# =========================
# CLEAN RULES
# =========================
required_cols = ["Incorrect Word", "Correct Word", "Karak Type"]

for col in required_cols:
    if col not in rules_df.columns:
        raise ValueError(f"Missing column in rules Excel: {col}")

rules_df = rules_df[required_cols].dropna()

for col in required_cols:
    rules_df[col] = rules_df[col].astype(str).str.strip()

rules_df = rules_df[
    (rules_df["Incorrect Word"] != "") &
    (rules_df["Correct Word"] != "") &
    (rules_df["Incorrect Word"] != rules_df["Correct Word"])
].copy()

rules_df = rules_df.drop_duplicates()

# =========================
# TARGET COUNT
# =========================
target_count = round(len(df) * KARAK_RATIO)

print("Dataset size:", len(df))
print("Target Karak errors:", target_count)

# =========================
# FIND ALL POSSIBLE MATCHES
# =========================
all_candidates = []

for rule_idx, row in rules_df.iterrows():

    correct = row["Correct Word"]
    incorrect = row["Incorrect Word"]
    karak_type = row["Karak Type"]

    pattern = r'(?<!\S)' + re.escape(correct) + r'(?!\S)'

    matching_indices = df[
        df["sentence"].str.contains(pattern, regex=True, na=False)
    ].index.tolist()

    for idx in matching_indices:
        all_candidates.append({
            "row_idx": idx,
            "correct_word": correct,
            "incorrect_word": incorrect,
            "karak_type": karak_type,
            "pattern": pattern
        })

print("Total possible Karak matches:", len(all_candidates))

if len(all_candidates) < target_count:
    print("WARNING: Available matches are less than target.")
    print("Will inject only:", len(all_candidates))
    target_count = len(all_candidates)

# =========================
# GROUP BY KARAK TYPE FOR UNIFORMITY
# =========================
type_to_candidates = defaultdict(list)

for cand in all_candidates:
    type_to_candidates[cand["karak_type"]].append(cand)

print("\nAvailable by Karak Type:")
for k, v in type_to_candidates.items():
    print(k, ":", len(v))

karak_types = sorted(type_to_candidates.keys())
num_types = len(karak_types)

base_quota = target_count // num_types
remainder = target_count % num_types

type_quota = {}

for i, ktype in enumerate(karak_types):
    type_quota[ktype] = base_quota + (1 if i < remainder else 0)

print("\nInitial target quota by Karak Type:")
for k, v in type_quota.items():
    print(k, ":", v)

# =========================
# SAMPLE UNIFORMLY WITHOUT DUPLICATING ROWS
# =========================
selected = []
used_rows = set()

# First pass: balanced selection by Karak Type
for ktype in karak_types:

    candidates = type_to_candidates[ktype]
    random.shuffle(candidates)

    needed = type_quota[ktype]
    count = 0

    for cand in candidates:
        if cand["row_idx"] not in used_rows:
            selected.append(cand)
            used_rows.add(cand["row_idx"])
            count += 1

        if count >= needed:
            break

# Second pass: fill remaining target from any type
remaining_needed = target_count - len(selected)

if remaining_needed > 0:
    print("\nFilling remaining:", remaining_needed)

    leftover = [
        cand for cand in all_candidates
        if cand["row_idx"] not in used_rows
    ]

    random.shuffle(leftover)

    for cand in leftover[:remaining_needed]:
        selected.append(cand)
        used_rows.add(cand["row_idx"])

print("\nFinal selected Karak injections:", len(selected))

# =========================
# INJECT ERRORS
# =========================
injection_records = []

for cand in selected:

    idx = cand["row_idx"]
    original_sentence = df.at[idx, "sentence"]

    new_sentence = re.sub(
        cand["pattern"],
        cand["incorrect_word"],
        original_sentence,
        count=1
    )

    df.at[idx, "sentence"] = new_sentence

    old_label = str(df.at[idx, "error_label"])

    if old_label == "0":
        new_label = KARAK_LABEL
    else:
        labels = [x.strip() for x in old_label.split(",") if x.strip()]
        if KARAK_LABEL not in labels:
            labels.append(KARAK_LABEL)
        labels = sorted(labels, key=lambda x: int(x) if x.isdigit() else x)
        new_label = ",".join(labels)

    df.at[idx, "error_label"] = new_label

    injection_records.append({
        "row_idx": idx,
        "karak_type": cand["karak_type"],
        "correct_word": cand["correct_word"],
        "incorrect_word": cand["incorrect_word"],
        "before_sentence": original_sentence,
        "after_sentence": new_sentence,
        "old_label": old_label,
        "new_label": new_label
    })

# =========================
# SAVE OUTPUTS
# =========================
df.to_csv(OUTPUT_PATH, index=False)

report_df = pd.DataFrame(injection_records)
report_df.to_csv(REPORT_PATH, index=False)

print("\nSaved injected dataset:")
print(OUTPUT_PATH)

print("\nSaved Karak injection report:")
print(REPORT_PATH)

# =========================
# SUMMARY
# =========================
print("\nKarak injection distribution:")
print(report_df["karak_type"].value_counts())

print("\nLabel distribution after Karak injection:")
print(df["error_label"].value_counts().sort_index())

print("\nExamples:")
pd.set_option("display.max_colwidth", None)
print(
    report_df[
        ["karak_type", "correct_word", "incorrect_word", "before_sentence", "after_sentence", "new_label"]
    ].head(20).to_string(index=False)
)

Dataset size: 51000
Target Karak errors: 2082
Total possible Karak matches: 2099

Available by Karak Type:
Kartā : 173
Karma : 254
Karan : 188
Sampradān : 135
Apādān : 165
Adhikaran : 1184

Initial target quota by Karak Type:
Adhikaran : 347
Apādān : 347
Karan : 347
Karma : 347
Kartā : 347
Sampradān : 347

Filling remaining: 849

Final selected Karak injections: 2041

Saved injected dataset:
/kaggle/working/51k_after_tense_pronoun_pratyaya_number_person_verb_punctuation_karak_injection.csv

Saved Karak injection report:
/kaggle/working/karak_injection_report.csv

Karak injection distribution:
karak_type
Adhikaran    1155
Karma         252
Karan         181
Kartā         166
Apādān        160
Sampradān     127
Name: count, dtype: int64

Label distribution after Karak injection:
error_label
0         36973
11         1753
2          3772
2,11        211
2,3          78
          ...  
7,11         65
7,9          62
7,9,11        2
9          1923
9,11        112
Name: count, Length: 93,

In [2]:
import pandas as pd
import random
import re

# =========================
# PATHS
# =========================
INPUT_PATH = "/kaggle/input/datasets/shruteesaha/51k-after-karak/51k_after_tense_pronoun_pratyaya_number_person_verb_punctuation_karak_injection.csv"

TRIAL_OUTPUT_PATH = "/kaggle/working/500_syntax_trial_dataset.csv"
TRIAL_EXAMPLES_PATH = "/kaggle/working/500_syntax_trial_examples.xlsx"

SYNTAX_LABEL = "10"
TRIAL_TARGET_COUNT = 500

random.seed(42)

# =========================
# LOAD
# =========================
df = pd.read_csv(INPUT_PATH)

df["sentence"] = df["sentence"].astype(str)
df["error_label"] = df["error_label"].astype(str)

# =========================
# HELPERS
# =========================
def add_label(old_label, new_label):
    old_label = str(old_label)

    if old_label == "0":
        return new_label

    labels = old_label.split(",")

    if new_label not in labels:
        labels.append(new_label)

    labels = sorted(labels, key=lambda x: int(x))
    return ",".join(labels)


def clean_spaces(sentence):
    sentence = re.sub(r"\s+", " ", sentence)
    sentence = re.sub(r"\s+([।,?!;:])", r"\1", sentence)
    return sentence.strip()


def split_punctuation(sentence):
    sentence = sentence.strip()

    if len(sentence) > 0 and sentence[-1] in "।?!":
        return sentence[:-1].strip(), sentence[-1]

    return sentence, ""


def is_safe_sentence(sentence):
    words = str(sentence).split()

    if len(words) < 6:
        return False

    if str(sentence).count(",") > 2:
        return False

    return True


# =========================
# RULE 1: FINAL VERB DISPLACEMENT
# =========================
def rule_final_verb_displacement(sentence):
    core, punct = split_punctuation(sentence)
    words = core.split()

    if len(words) < 6:
        return None

    phrase_len = random.choice([1, 2])

    if len(words) <= phrase_len + 3:
        return None

    prefix = words[:-phrase_len-1]
    phrase = words[-phrase_len-1:-1]
    verb = words[-1:]

    if len(prefix) < 2:
        return None

    new_words = prefix + verb + phrase
    new_sentence = clean_spaces(" ".join(new_words) + punct)

    if new_sentence != sentence:
        return new_sentence

    return None


# =========================
# RULE 2: MODIFIER-NOUN SWAP
# =========================
def rule_modifier_noun_swap(sentence):
    core, punct = split_punctuation(sentence)
    words = core.split()

    if len(words) < 6:
        return None

    modifier_suffixes = (
        "ীয়া", "িয়া", "ৰীয়া", "জাতীয়", "স্থানীয়",
        "সামাজিক", "ৰাজনৈতিক", "ধাৰ্মিক", "সাংস্কৃতিক",
        "ঐতিহাসিক", "বৈজ্ঞানিক", "প্ৰাকৃতিক", "ভৌগোলিক",
        "মানসিক", "শাৰীৰিক", "মূলক", "সম্পৰ্কীয়", "বিষয়ক"
    )

    candidate_positions = []

    for i in range(len(words) - 1):
        if words[i].endswith(modifier_suffixes):
            candidate_positions.append(i)

    if not candidate_positions:
        return None

    i = random.choice(candidate_positions)

    new_words = words.copy()
    new_words[i], new_words[i + 1] = new_words[i + 1], new_words[i]

    new_sentence = clean_spaces(" ".join(new_words) + punct)

    if new_sentence != sentence:
        return new_sentence

    return None


# =========================
# RULE 3: TEMPORAL PHRASE DISPLACEMENT
# =========================
def rule_temporal_phrase_displacement(sentence):
    core, punct = split_punctuation(sentence)
    words = core.split()

    if len(words) < 7:
        return None

    temporal_markers = (
        "চনত", "চনতে", "মাহত", "দিনা", "সময়ত",
        "পিছত", "আগতে", "কালত", "বছৰত"
    )

    candidate_positions = []

    for i, word in enumerate(words):
        if word.endswith(temporal_markers):
            candidate_positions.append(i)

    if not candidate_positions:
        return None

    i = random.choice(candidate_positions)

    start = max(0, i - 1)
    end = i + 1

    temporal_phrase = words[start:end]
    remaining = words[:start] + words[end:]

    if len(remaining) < 5:
        return None

    insert_pos = len(remaining) - 1

    if insert_pos <= 1:
        return None

    new_words = remaining[:insert_pos] + temporal_phrase + remaining[insert_pos:]
    new_sentence = clean_spaces(" ".join(new_words) + punct)

    if new_sentence != sentence:
        return new_sentence

    return None


# =========================
# APPLY SYNTAX RULE
# =========================
def inject_syntax_error(sentence):
    if not is_safe_sentence(sentence):
        return None, None

    rules = [
        ("final_verb_displacement", rule_final_verb_displacement),
        ("modifier_noun_swap", rule_modifier_noun_swap),
        ("temporal_phrase_displacement", rule_temporal_phrase_displacement)
    ]

    random.shuffle(rules)

    for rule_name, rule_func in rules:
        new_sentence = rule_func(sentence)

        if new_sentence and new_sentence != sentence:
            return new_sentence, rule_name

    return None, None


# =========================
# TRIAL INJECTION
# =========================
trial_df = df.copy()

candidate_indices = trial_df.index[
    trial_df["sentence"].apply(is_safe_sentence)
].tolist()

random.shuffle(candidate_indices)

injected = 0
examples = []

for idx in candidate_indices:
    if injected >= TRIAL_TARGET_COUNT:
        break

    original_sentence = trial_df.at[idx, "sentence"]
    new_sentence, rule_name = inject_syntax_error(original_sentence)

    if new_sentence is None:
        continue

    trial_df.at[idx, "sentence"] = new_sentence
    trial_df.at[idx, "error_label"] = add_label(trial_df.at[idx, "error_label"], SYNTAX_LABEL)

    examples.append({
        "index": idx,
        "rule_used": rule_name,
        "before_sentence": original_sentence,
        "after_sentence": new_sentence,
        "error_label": trial_df.at[idx, "error_label"]
    })

    injected += 1

# =========================
# SAVE ONLY TRIAL VERSION
# =========================
trial_df.to_csv(TRIAL_OUTPUT_PATH, index=False, encoding="utf-8-sig")

examples_df = pd.DataFrame(examples)
examples_df.to_excel(TRIAL_EXAMPLES_PATH, index=False)

print("Trial syntax errors injected:", injected)
print("Trial dataset saved to:", TRIAL_OUTPUT_PATH)
print("Trial examples saved to:", TRIAL_EXAMPLES_PATH)

print("\nRule-wise count:")
print(examples_df["rule_used"].value_counts())

print("\nFirst 30 examples:")
for ex in examples[:30]:
    print("Index :", ex["index"])
    print("Rule  :", ex["rule_used"])
    print("Before:", ex["before_sentence"])
    print("After :", ex["after_sentence"])
    print("Label :", ex["error_label"])
    print("-" * 100)

Trial syntax errors injected: 500
Trial dataset saved to: /kaggle/working/500_syntax_trial_dataset.csv
Trial examples saved to: /kaggle/working/500_syntax_trial_examples.xlsx

Rule-wise count:
rule_used
final_verb_displacement         439
temporal_phrase_displacement     32
modifier_noun_swap               29
Name: count, dtype: int64

First 30 examples:
Index : 5208
Rule  : final_verb_displacement
Before: ২০০০-২০০১ ক্ৰীড়াবৰ্ষৰ নেশ্যনেল ফুটবল লীগ ইষ্ট বেংগল ফুটবল ক্লাবে জিকে ।
After : ২০০০-২০০১ ক্ৰীড়াবৰ্ষৰ নেশ্যনেল ফুটবল লীগ ইষ্ট বেংগল ফুটবল জিকে ক্লাবে।
Label : 10
----------------------------------------------------------------------------------------------------
Index : 49092
Rule  : final_verb_displacement
Before: এইখনেই পাছলৈ সাম্যবাদী ইস্তাহাৰ বা কমিউনিষ্ট মেনিফেষ্টো নামেৰে জনাজাত হল।
After : এইখনেই পাছলৈ সাম্যবাদী ইস্তাহাৰ বা কমিউনিষ্ট মেনিফেষ্টো নামেৰে হল জনাজাত।
Label : 10
----------------------------------------------------------------------------------------------------
Ind

In [3]:
import pandas as pd
import random
import re

# =========================
# PATHS
# =========================
INPUT_PATH = "/kaggle/input/datasets/shruteesaha/51k-after-karak/51k_after_tense_pronoun_pratyaya_number_person_verb_punctuation_karak_injection.csv"

TRIAL_OUTPUT_PATH = "/kaggle/working/500_syntax_trial_dataset_v2.csv"
TRIAL_EXAMPLES_PATH = "/kaggle/working/500_syntax_trial_examples_v2.xlsx"

SYNTAX_LABEL = "10"   # your syntax label seems to be 10 from the output
TRIAL_TARGET_COUNT = 500

random.seed(43)

# =========================
# LOAD
# =========================
df = pd.read_csv(INPUT_PATH)

df["sentence"] = df["sentence"].astype(str)
df["error_label"] = df["error_label"].astype(str)

# =========================
# HELPERS
# =========================
def add_label(old_label, new_label):
    old_label = str(old_label)

    if old_label == "0":
        return new_label

    labels = old_label.split(",")

    if new_label not in labels:
        labels.append(new_label)

    labels = sorted(labels, key=lambda x: int(x))
    return ",".join(labels)


def clean_spaces(sentence):
    sentence = re.sub(r"\s+", " ", sentence)
    sentence = re.sub(r"\s+([।,?!;:])", r"\1", sentence)
    return sentence.strip()


def split_punctuation(sentence):
    sentence = sentence.strip()

    if len(sentence) > 0 and sentence[-1] in "।?!":
        return sentence[:-1].strip(), sentence[-1]

    return sentence, ""


def is_safe_sentence(sentence):
    sentence = str(sentence).strip()
    words = sentence.split()

    if len(words) < 6:
        return False

    if sentence.count(",") > 2:
        return False

    # avoid quoted/dialogue endings
    if "”" in sentence or "“" in sentence or '"' in sentence:
        return False

    return True


# =========================
# COMMON VERB / AUX WORDS
# =========================
AUX_WORDS = {
    "হয়", "হৈছে", "হৈছিল", "হল", "হ'ল", "হ’ব",
    "আছে", "আছিল", "থাকে", "থাকিছিল",
    "কৰিছে", "কৰিছিল", "কৰে", "কৰিলে", "কৰা",
    "দিয়ে", "দিলে", "পায়", "পালে",
    "লাভ", "প্ৰদান", "যোগদান", "বহন", "স্থাপন",
    "পৰিচালিত", "প্ৰতিষ্ঠিত", "বিবেচিত", "লিপ্ত",
    "জনাজাত", "নিৰ্বাচিত"
}

# common compound predicate left parts
COMPOUND_LEFT = {
    "লাভ", "প্ৰদান", "যোগদান", "বহন", "স্থাপন",
    "পৰিচালিত", "প্ৰতিষ্ঠিত", "বিবেচিত", "লিপ্ত",
    "জনাজাত", "নিৰ্বাচিত", "উদ্বোধন", "প্ৰকাশ",
    "ব্যৱহাৰ", "আৰম্ভ", "সৃষ্টি", "নিৰ্মাণ"
}

# common auxiliary right parts
AUX_RIGHT = {
    "কৰে", "কৰিছে", "কৰিছিল", "কৰিলে", "কৰা",
    "হয়", "হৈছে", "হৈছিল", "হল", "হ'ল",
    "আছে", "আছিল"
}


def ends_with_compound_verb(words):
    """
    Detects endings like:
    স্থাপন কৰা হৈছিল
    প্ৰদান কৰিছে
    যোগদান কৰে
    লিপ্ত হয়
    """
    if len(words) < 2:
        return False

    if words[-2] in COMPOUND_LEFT and words[-1] in AUX_RIGHT:
        return True

    if len(words) >= 3:
        if words[-3] in COMPOUND_LEFT and words[-2] in AUX_RIGHT and words[-1] in AUX_RIGHT:
            return True

    return False


# =========================
# RULE 1: CLEAN PREDICATE FRONTING
# =========================
def rule_clean_predicate_fronting(sentence):
    """
    Moves the whole final predicate before the phrase before it.
    Does not split compound verbs.

    Example:
    তেওঁ অধ্যক্ষ আছিল।
    -> তেওঁ আছিল অধ্যক্ষ।
    """

    core, punct = split_punctuation(sentence)
    words = core.split()

    if len(words) < 6:
        return None

    # Decide final predicate size
    pred_len = 1

    if len(words) >= 2 and words[-2] in COMPOUND_LEFT and words[-1] in AUX_RIGHT:
        pred_len = 2

    if len(words) >= 3 and words[-3] in COMPOUND_LEFT and words[-2] in AUX_RIGHT and words[-1] in AUX_RIGHT:
        pred_len = 3

    phrase_len = random.choice([1, 2])

    if len(words) <= pred_len + phrase_len + 3:
        return None

    prefix = words[:-(pred_len + phrase_len)]
    phrase = words[-(pred_len + phrase_len):-pred_len]
    predicate = words[-pred_len:]

    if len(prefix) < 2:
        return None

    # Avoid weird cases where phrase itself is verbal
    if any(w in AUX_WORDS for w in phrase):
        return None

    new_words = prefix + predicate + phrase
    new_sentence = clean_spaces(" ".join(new_words) + punct)

    if new_sentence != sentence:
        return new_sentence

    return None


# =========================
# RULE 2: OBJECT-PREDICATE INVERSION
# =========================
def rule_object_predicate_inversion(sentence):
    """
    Specifically targets compound predicates:
    যোগদান কৰে -> কৰে যোগদান
    প্ৰতিষ্ঠিত কৰিছে -> কৰিছে প্ৰতিষ্ঠিত

    This produces common word-order style syntax error without changing word form.
    """

    core, punct = split_punctuation(sentence)
    words = core.split()

    if len(words) < 6:
        return None

    candidate_positions = []

    for i in range(len(words) - 1):
        if words[i] in COMPOUND_LEFT and words[i + 1] in AUX_RIGHT:
            candidate_positions.append(i)

    if not candidate_positions:
        return None

    i = random.choice(candidate_positions)

    # avoid already sentence-final tiny fragments if too close to end with no context
    if i < 2:
        return None

    new_words = words.copy()
    new_words[i], new_words[i + 1] = new_words[i + 1], new_words[i]

    new_sentence = clean_spaces(" ".join(new_words) + punct)

    if new_sentence != sentence:
        return new_sentence

    return None


# =========================
# RULE 3: MODIFIER-NOUN SWAP
# =========================
def rule_modifier_noun_swap(sentence):
    core, punct = split_punctuation(sentence)
    words = core.split()

    if len(words) < 6:
        return None

    modifier_suffixes = (
        "ীয়া", "িয়া", "ৰীয়া", "জাতীয়", "স্থানীয়",
        "সামাজিক", "ৰাজনৈতিক", "ধাৰ্মিক", "সাংস্কৃতিক",
        "ঐতিহাসিক", "বৈজ্ঞানিক", "প্ৰাকৃতিক", "ভৌগোলিক",
        "মানসিক", "শাৰীৰিক", "মূলক", "সম্পৰ্কীয়", "বিষয়ক"
    )

    candidate_positions = []

    for i in range(len(words) - 1):
        if words[i].endswith(modifier_suffixes):
            # avoid swapping with punctuation-ish token
            if re.search(r"[।,?!;:]", words[i + 1]):
                continue
            candidate_positions.append(i)

    if not candidate_positions:
        return None

    i = random.choice(candidate_positions)

    new_words = words.copy()
    new_words[i], new_words[i + 1] = new_words[i + 1], new_words[i]

    new_sentence = clean_spaces(" ".join(new_words) + punct)

    if new_sentence != sentence:
        return new_sentence

    return None


# =========================
# RULE 4: TEMPORAL PHRASE DISPLACEMENT
# =========================
def rule_temporal_phrase_displacement(sentence):
    core, punct = split_punctuation(sentence)
    words = core.split()

    if len(words) < 7:
        return None

    temporal_markers = (
        "চনত", "চনতে", "মাহত", "দিনা", "সময়ত",
        "পিছত", "আগতে", "কালত", "বছৰত"
    )

    candidate_positions = []

    for i, word in enumerate(words):
        if word.endswith(temporal_markers):
            candidate_positions.append(i)

    if not candidate_positions:
        return None

    i = random.choice(candidate_positions)

    start = max(0, i - 1)
    end = i + 1

    temporal_phrase = words[start:end]
    remaining = words[:start] + words[end:]

    if len(remaining) < 5:
        return None

    insert_pos = len(remaining) - 1

    if insert_pos <= 1:
        return None

    new_words = remaining[:insert_pos] + temporal_phrase + remaining[insert_pos:]
    new_sentence = clean_spaces(" ".join(new_words) + punct)

    if new_sentence != sentence:
        return new_sentence

    return None


# =========================
# APPLY SYNTAX RULE
# =========================
def inject_syntax_error(sentence):
    if not is_safe_sentence(sentence):
        return None, None

    rules = [
        ("object_predicate_inversion", rule_object_predicate_inversion),
        ("clean_predicate_fronting", rule_clean_predicate_fronting),
        ("modifier_noun_swap", rule_modifier_noun_swap),
        ("temporal_phrase_displacement", rule_temporal_phrase_displacement)
    ]

    random.shuffle(rules)

    for rule_name, rule_func in rules:
        new_sentence = rule_func(sentence)

        if new_sentence and new_sentence != sentence:
            return new_sentence, rule_name

    return None, None


# =========================
# TRIAL INJECTION
# =========================
trial_df = df.copy()

candidate_indices = trial_df.index[
    trial_df["sentence"].apply(is_safe_sentence)
].tolist()

random.shuffle(candidate_indices)

injected = 0
examples = []

for idx in candidate_indices:
    if injected >= TRIAL_TARGET_COUNT:
        break

    original_sentence = trial_df.at[idx, "sentence"]
    new_sentence, rule_name = inject_syntax_error(original_sentence)

    if new_sentence is None:
        continue

    trial_df.at[idx, "sentence"] = new_sentence
    trial_df.at[idx, "error_label"] = add_label(trial_df.at[idx, "error_label"], SYNTAX_LABEL)

    examples.append({
        "index": idx,
        "rule_used": rule_name,
        "before_sentence": original_sentence,
        "after_sentence": new_sentence,
        "error_label": trial_df.at[idx, "error_label"]
    })

    injected += 1

# =========================
# SAVE TRIAL VERSION
# =========================
trial_df.to_csv(TRIAL_OUTPUT_PATH, index=False, encoding="utf-8-sig")

examples_df = pd.DataFrame(examples)
examples_df.to_excel(TRIAL_EXAMPLES_PATH, index=False)

print("Trial syntax errors injected:", injected)
print("Trial dataset saved to:", TRIAL_OUTPUT_PATH)
print("Trial examples saved to:", TRIAL_EXAMPLES_PATH)

print("\nRule-wise count:")
print(examples_df["rule_used"].value_counts())

print("\nFirst 40 examples:")
for ex in examples[:40]:
    print("Index :", ex["index"])
    print("Rule  :", ex["rule_used"])
    print("Before:", ex["before_sentence"])
    print("After :", ex["after_sentence"])
    print("Label :", ex["error_label"])
    print("-" * 100)

Trial syntax errors injected: 500
Trial dataset saved to: /kaggle/working/500_syntax_trial_dataset_v2.csv
Trial examples saved to: /kaggle/working/500_syntax_trial_examples_v2.xlsx

Rule-wise count:
rule_used
clean_predicate_fronting        399
temporal_phrase_displacement     36
modifier_noun_swap               34
object_predicate_inversion       31
Name: count, dtype: int64

First 40 examples:
Index : 8092
Rule  : clean_predicate_fronting
Before: ডৌডৌ শব্দটো পৰ্তুগীজ শব্দ দ’ওদ’ৰ পৰা আহিছে, যাৰ অৰ্থ মুৰ্খ।
After : ডৌডৌ শব্দটো পৰ্তুগীজ শব্দ দ’ওদ’ৰ পৰা আহিছে, যাৰ মুৰ্খ অৰ্থ।
Label : 10
----------------------------------------------------------------------------------------------------
Index : 23194
Rule  : clean_predicate_fronting
Before: ১৯৭৮, ১৯৮০ আৰু ১৯৮৫ চনত ইষ্ট বেংগলে ফেডাৰেচন কাপ দখল কৰে।
After : ১৯৭৮, ১৯৮০ আৰু ১৯৮৫ চনত ইষ্ট বেংগলে ফেডাৰেচন কৰে কাপ দখল।
Label : 10
----------------------------------------------------------------------------------------------------
Index : 12858
Ru

In [4]:
import pandas as pd
import random
import re

# =========================
# PATHS
# =========================
INPUT_PATH = "/kaggle/input/datasets/shruteesaha/51k-after-karak/51k_after_tense_pronoun_pratyaya_number_person_verb_punctuation_karak_injection.csv"

TRIAL_OUTPUT_PATH = "/kaggle/working/500_syntax_trial_dataset_v3.csv"
TRIAL_EXAMPLES_PATH = "/kaggle/working/500_syntax_trial_examples_v3.xlsx"

SYNTAX_LABEL = "10"
TRIAL_TARGET_COUNT = 500

random.seed(44)

# =========================
# LOAD
# =========================
df = pd.read_csv(INPUT_PATH)

df["sentence"] = df["sentence"].astype(str)
df["error_label"] = df["error_label"].astype(str)

# =========================
# HELPERS
# =========================
def add_label(old_label, new_label):
    old_label = str(old_label)

    if old_label == "0":
        return new_label

    labels = old_label.split(",")

    if new_label not in labels:
        labels.append(new_label)

    labels = sorted(labels, key=lambda x: int(x))
    return ",".join(labels)


def clean_spaces(sentence):
    sentence = re.sub(r"\s+", " ", sentence)
    sentence = re.sub(r"\s+([।,?!;:])", r"\1", sentence)
    return sentence.strip()


def split_punctuation(sentence):
    sentence = sentence.strip()

    if len(sentence) > 0 and sentence[-1] in "।?!":
        return sentence[:-1].strip(), sentence[-1]

    return sentence, ""


def is_safe_sentence(sentence):
    sentence = str(sentence).strip()
    words = sentence.split()

    if len(words) < 6:
        return False

    if sentence.count(",") > 2:
        return False

    if "”" in sentence or "“" in sentence or '"' in sentence:
        return False

    if "(" in sentence or ")" in sentence:
        return False

    return True


# =========================
# CONTROLLED COMPOUND PREDICATES
# =========================
COMPOUND_LEFT = {
    "লাভ", "প্ৰদান", "যোগদান", "বহন", "স্থাপন",
    "পৰিচালনা", "প্ৰতিষ্ঠা", "প্ৰকাশ", "ব্যৱহাৰ",
    "আৰম্ভ", "সৃষ্টি", "নিৰ্মাণ", "উদ্বোধন",
    "পালন", "গ্ৰহণ", "উল্লেখ", "ৰচনা",
    "অলংকৃত", "দখল", "আৰোহণ"
}

AUX_RIGHT = {
    "কৰে", "কৰিছে", "কৰিছিল", "কৰিলে", "কৰা",
    "হয়", "হৈছে", "হৈছিল", "হল", "হ'ল"
}


# =========================
# RULE 1: OBJECT-PREDICATE INVERSION
# =========================
def rule_object_predicate_inversion(sentence):
    """
    Controlled inversion:
    যোগদান কৰে -> কৰে যোগদান
    দখল কৰে -> কৰে দখল

    This preserves all word forms.
    """

    core, punct = split_punctuation(sentence)
    words = core.split()

    if len(words) < 6:
        return None

    candidate_positions = []

    for i in range(len(words) - 1):
        if words[i] in COMPOUND_LEFT and words[i + 1] in AUX_RIGHT:
            if i >= 2:
                candidate_positions.append(i)

    if not candidate_positions:
        return None

    i = random.choice(candidate_positions)

    new_words = words.copy()
    new_words[i], new_words[i + 1] = new_words[i + 1], new_words[i]

    new_sentence = clean_spaces(" ".join(new_words) + punct)

    if new_sentence != sentence:
        return new_sentence

    return None


# =========================
# RULE 2: MODIFIER-NOUN SWAP
# =========================
def rule_modifier_noun_swap(sentence):
    """
    Modifier + noun -> noun + modifier
    Example:
    হালধীয়া কাপোৰ -> কাপোৰ হালধীয়া
    """

    core, punct = split_punctuation(sentence)
    words = core.split()

    if len(words) < 6:
        return None

    modifier_suffixes = (
        "ীয়া", "িয়া", "ৰীয়া", "জাতীয়", "স্থানীয়",
        "সামাজিক", "ৰাজনৈতিক", "ধাৰ্মিক", "সাংস্কৃতিক",
        "ঐতিহাসিক", "বৈজ্ঞানিক", "প্ৰাকৃতিক", "ভৌগোলিক",
        "মানসিক", "শাৰীৰিক", "মূলক", "সম্পৰ্কীয়", "বিষয়ক",
        "হালধীয়া", "কলা", "বগা", "ৰঙা", "নীলা"
    )

    candidate_positions = []

    for i in range(len(words) - 1):
        if words[i].endswith(modifier_suffixes):
            if re.search(r"[।,?!;:]", words[i + 1]):
                continue

            # avoid swapping with verb/aux
            if words[i + 1] in AUX_RIGHT:
                continue

            candidate_positions.append(i)

    if not candidate_positions:
        return None

    i = random.choice(candidate_positions)

    new_words = words.copy()
    new_words[i], new_words[i + 1] = new_words[i + 1], new_words[i]

    new_sentence = clean_spaces(" ".join(new_words) + punct)

    if new_sentence != sentence:
        return new_sentence

    return None


# =========================
# RULE 3: STRICT TEMPORAL PHRASE DISPLACEMENT
# =========================
def rule_temporal_phrase_displacement(sentence):
    """
    Moves a sentence-initial temporal phrase before the final predicate.

    Example:
    ১৯৫২ চনত তেওঁ আঘাতপ্ৰাপ্ত হয়।
    -> তেওঁ আঘাতপ্ৰাপ্ত ১৯৫২ চনত হয়।
    """

    core, punct = split_punctuation(sentence)
    words = core.split()

    if len(words) < 7:
        return None

    temporal_markers = (
        "চনত", "চনতে", "মাহত", "দিনা", "সময়ত",
        "পিছত", "আগতে", "কালত", "বছৰত"
    )

    # only use temporal phrase if it appears near sentence beginning
    candidate_positions = []

    for i, word in enumerate(words[:5]):
        if word.endswith(temporal_markers):
            candidate_positions.append(i)

    if not candidate_positions:
        return None

    i = random.choice(candidate_positions)

    start = max(0, i - 1)
    end = i + 1

    temporal_phrase = words[start:end]
    remaining = words[:start] + words[end:]

    if len(remaining) < 5:
        return None

    # avoid moving if remaining has another sentence boundary
    if any("।" in w for w in remaining):
        return None

    insert_pos = len(remaining) - 1

    if insert_pos <= 2:
        return None

    # avoid putting temporal phrase between auxiliary compounds
    if remaining[insert_pos - 1] in COMPOUND_LEFT and remaining[insert_pos] in AUX_RIGHT:
        return None

    new_words = remaining[:insert_pos] + temporal_phrase + remaining[insert_pos:]
    new_sentence = clean_spaces(" ".join(new_words) + punct)

    if new_sentence != sentence:
        return new_sentence

    return None


# =========================
# APPLY SYNTAX RULE
# =========================
def inject_syntax_error(sentence):
    if not is_safe_sentence(sentence):
        return None, None

    rules = [
        ("object_predicate_inversion", rule_object_predicate_inversion),
        ("modifier_noun_swap", rule_modifier_noun_swap),
        ("temporal_phrase_displacement", rule_temporal_phrase_displacement)
    ]

    random.shuffle(rules)

    for rule_name, rule_func in rules:
        new_sentence = rule_func(sentence)

        if new_sentence and new_sentence != sentence:
            return new_sentence, rule_name

    return None, None


# =========================
# TRIAL INJECTION
# =========================
trial_df = df.copy()

candidate_indices = trial_df.index[
    trial_df["sentence"].apply(is_safe_sentence)
].tolist()

random.shuffle(candidate_indices)

injected = 0
examples = []

for idx in candidate_indices:
    if injected >= TRIAL_TARGET_COUNT:
        break

    original_sentence = trial_df.at[idx, "sentence"]
    new_sentence, rule_name = inject_syntax_error(original_sentence)

    if new_sentence is None:
        continue

    trial_df.at[idx, "sentence"] = new_sentence
    trial_df.at[idx, "error_label"] = add_label(trial_df.at[idx, "error_label"], SYNTAX_LABEL)

    examples.append({
        "index": idx,
        "rule_used": rule_name,
        "before_sentence": original_sentence,
        "after_sentence": new_sentence,
        "error_label": trial_df.at[idx, "error_label"]
    })

    injected += 1

# =========================
# SAVE TRIAL VERSION
# =========================
trial_df.to_csv(TRIAL_OUTPUT_PATH, index=False, encoding="utf-8-sig")

examples_df = pd.DataFrame(examples)
examples_df.to_excel(TRIAL_EXAMPLES_PATH, index=False)

print("Trial syntax errors injected:", injected)
print("Trial dataset saved to:", TRIAL_OUTPUT_PATH)
print("Trial examples saved to:", TRIAL_EXAMPLES_PATH)

print("\nRule-wise count:")
print(examples_df["rule_used"].value_counts())

print("\nFirst 40 examples:")
for ex in examples[:40]:
    print("Index :", ex["index"])
    print("Rule  :", ex["rule_used"])
    print("Before:", ex["before_sentence"])
    print("After :", ex["after_sentence"])
    print("Label :", ex["error_label"])
    print("-" * 100)

Trial syntax errors injected: 500
Trial dataset saved to: /kaggle/working/500_syntax_trial_dataset_v3.csv
Trial examples saved to: /kaggle/working/500_syntax_trial_examples_v3.xlsx

Rule-wise count:
rule_used
object_predicate_inversion      192
modifier_noun_swap              175
temporal_phrase_displacement    133
Name: count, dtype: int64

First 40 examples:
Index : 29939
Rule  : object_predicate_inversion
Before: উত্তৰ ভাৰতৰ অন্যতম জনপ্ৰিয় উৎসৱ ছঠ ঘাইকৈ বিহাৰ, উত্তৰ প্ৰদেশ আদিত পালন কৰা হয় যদিও আজিকালি প্ৰায় সকলো ঠাইতে ইয়াৰ উদযাপন কৰা দেখা যায়।
After : উত্তৰ ভাৰতৰ অন্যতম জনপ্ৰিয় উৎসৱ ছঠ ঘাইকৈ বিহাৰ, উত্তৰ প্ৰদেশ আদিত কৰা পালন হয় যদিও আজিকালি প্ৰায় সকলো ঠাইতে ইয়াৰ উদযাপন কৰা দেখা যায়।
Label : 10
----------------------------------------------------------------------------------------------------
Index : 42212
Rule  : modifier_noun_swap
Before: বিশেষ কেইটামান কাৰণত সমুদ্ৰ-তৰংগ যেতিয়া উপকূলভাগত আছাৰ খাবলৈ পৰি মহাতংৰগলৈ ৰূপান্তৰিত হয়,তাকে ছুনামি কোৱা হয়।
After : বিশেষ কেইটাম

In [5]:
import pandas as pd
import random
import re

# =========================
# PATHS
# =========================
INPUT_PATH = "/kaggle/input/datasets/shruteesaha/51k-after-karak/51k_after_tense_pronoun_pratyaya_number_person_verb_punctuation_karak_injection.csv"

OUTPUT_PATH = "/kaggle/working/51k_after_syntax_injection.csv"
EXAMPLES_PATH = "/kaggle/working/syntax_error_examples_full.xlsx"

SYNTAX_LABEL = "10"
TARGET_COUNT = 18292

random.seed(44)

# =========================
# LOAD
# =========================
df = pd.read_csv(INPUT_PATH)

df["sentence"] = df["sentence"].astype(str)
df["error_label"] = df["error_label"].astype(str)

# =========================
# HELPERS
# =========================
def add_label(old_label, new_label):
    old_label = str(old_label)

    if old_label == "0":
        return new_label

    labels = old_label.split(",")

    if new_label not in labels:
        labels.append(new_label)

    labels = sorted(labels, key=lambda x: int(x))
    return ",".join(labels)


def clean_spaces(sentence):
    sentence = re.sub(r"\s+", " ", sentence)
    sentence = re.sub(r"\s+([।,?!;:])", r"\1", sentence)
    return sentence.strip()


def split_punctuation(sentence):
    sentence = sentence.strip()

    if len(sentence) > 0 and sentence[-1] in "।?!":
        return sentence[:-1].strip(), sentence[-1]

    return sentence, ""


def is_safe_sentence(sentence):
    sentence = str(sentence).strip()
    words = sentence.split()

    if len(words) < 6:
        return False

    # stricter filter for complex sentences
    if sentence.count(",") > 1:
        return False

    if "”" in sentence or "“" in sentence or '"' in sentence:
        return False

    if "(" in sentence or ")" in sentence:
        return False

    return True


# =========================
# RULE SET
# =========================
COMPOUND_LEFT = {
    "লাভ", "প্ৰদান", "যোগদান", "বহন", "স্থাপন",
    "পৰিচালনা", "প্ৰতিষ্ঠা", "প্ৰকাশ", "ব্যৱহাৰ",
    "আৰম্ভ", "সৃষ্টি", "নিৰ্মাণ", "উদ্বোধন",
    "পালন", "গ্ৰহণ", "উল্লেখ", "ৰচনা",
    "অলংকৃত", "দখল", "আৰোহণ"
}

AUX_RIGHT = {
    "কৰে", "কৰিছে", "কৰিছিল", "কৰিলে", "কৰা",
    "হয়", "হৈছে", "হৈছিল", "হল", "হ'ল"
}


# =========================
# RULE 1: OBJECT-PREDICATE INVERSION
# =========================
def rule_object_predicate_inversion(sentence):
    core, punct = split_punctuation(sentence)
    words = core.split()

    if len(words) < 6:
        return None

    candidate_positions = []

    for i in range(len(words) - 1):
        if words[i] in COMPOUND_LEFT and words[i + 1] in AUX_RIGHT:
            if i >= 2:
                candidate_positions.append(i)

    if not candidate_positions:
        return None

    i = random.choice(candidate_positions)

    new_words = words.copy()
    new_words[i], new_words[i + 1] = new_words[i + 1], new_words[i]

    new_sentence = clean_spaces(" ".join(new_words) + punct)

    if new_sentence != sentence:
        return new_sentence

    return None


# =========================
# RULE 2: MODIFIER-NOUN SWAP
# =========================
def rule_modifier_noun_swap(sentence):
    core, punct = split_punctuation(sentence)
    words = core.split()

    if len(words) < 6:
        return None

    modifier_suffixes = (
        "ীয়া", "িয়া", "ৰীয়া", "জাতীয়", "স্থানীয়",
        "সামাজিক", "ৰাজনৈতিক", "ধাৰ্মিক", "সাংস্কৃতিক",
        "ঐতিহাসিক", "বৈজ্ঞানিক", "প্ৰাকৃতিক", "ভৌগোলিক",
        "মানসিক", "শাৰীৰিক", "মূলক", "সম্পৰ্কীয়", "বিষয়ক",
        "হালধীয়া", "কলা", "বগা", "ৰঙা", "নীলা"
    )

    candidate_positions = []

    for i in range(len(words) - 1):
        if words[i].endswith(modifier_suffixes):
            if re.search(r"[।,?!;:]", words[i + 1]):
                continue

            if words[i + 1] in AUX_RIGHT:
                continue

            candidate_positions.append(i)

    if not candidate_positions:
        return None

    i = random.choice(candidate_positions)

    new_words = words.copy()
    new_words[i], new_words[i + 1] = new_words[i + 1], new_words[i]

    new_sentence = clean_spaces(" ".join(new_words) + punct)

    if new_sentence != sentence:
        return new_sentence

    return None


# =========================
# RULE 3: TEMPORAL PHRASE DISPLACEMENT
# =========================
def rule_temporal_phrase_displacement(sentence):
    core, punct = split_punctuation(sentence)
    words = core.split()

    if len(words) < 7:
        return None

    temporal_markers = (
        "চনত", "চনতে", "মাহত", "দিনা", "সময়ত",
        "পিছত", "আগতে", "কালত", "বছৰত"
    )

    candidate_positions = []

    for i, word in enumerate(words[:5]):
        if word.endswith(temporal_markers):
            candidate_positions.append(i)

    if not candidate_positions:
        return None

    i = random.choice(candidate_positions)

    start = max(0, i - 1)
    end = i + 1

    temporal_phrase = words[start:end]
    remaining = words[:start] + words[end:]

    if len(remaining) < 5:
        return None

    if any("।" in w for w in remaining):
        return None

    insert_pos = len(remaining) - 1

    if insert_pos <= 2:
        return None

    if remaining[insert_pos - 1] in COMPOUND_LEFT and remaining[insert_pos] in AUX_RIGHT:
        return None

    new_words = remaining[:insert_pos] + temporal_phrase + remaining[insert_pos:]
    new_sentence = clean_spaces(" ".join(new_words) + punct)

    if new_sentence != sentence:
        return new_sentence

    return None


# =========================
# APPLY SYNTAX RULE
# =========================
def inject_syntax_error(sentence):
    if not is_safe_sentence(sentence):
        return None, None

    rules = [
        ("object_predicate_inversion", rule_object_predicate_inversion),
        ("modifier_noun_swap", rule_modifier_noun_swap),
        ("temporal_phrase_displacement", rule_temporal_phrase_displacement)
    ]

    random.shuffle(rules)

    for rule_name, rule_func in rules:
        new_sentence = rule_func(sentence)

        if new_sentence and new_sentence != sentence:
            return new_sentence, rule_name

    return None, None


# =========================
# FULL INJECTION
# =========================
candidate_indices = df.index[
    df["sentence"].apply(is_safe_sentence)
].tolist()

random.shuffle(candidate_indices)

injected = 0
examples = []

for idx in candidate_indices:
    if injected >= TARGET_COUNT:
        break

    original_sentence = df.at[idx, "sentence"]
    new_sentence, rule_name = inject_syntax_error(original_sentence)

    if new_sentence is None:
        continue

    df.at[idx, "sentence"] = new_sentence
    df.at[idx, "error_label"] = add_label(df.at[idx, "error_label"], SYNTAX_LABEL)

    examples.append({
        "index": idx,
        "rule_used": rule_name,
        "before_sentence": original_sentence,
        "after_sentence": new_sentence,
        "error_label": df.at[idx, "error_label"]
    })

    injected += 1

# =========================
# SAVE
# =========================
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

examples_df = pd.DataFrame(examples)
examples_df.to_excel(EXAMPLES_PATH, index=False)

print("Syntax errors injected:", injected)
print("Target syntax errors:", TARGET_COUNT)
print("Output saved to:", OUTPUT_PATH)
print("Examples saved to:", EXAMPLES_PATH)

print("\nRule-wise count:")
print(examples_df["rule_used"].value_counts())

print("\nLabel distribution:")
print(df["error_label"].value_counts().sort_index())

print("\nFirst 50 examples:")
for ex in examples[:50]:
    print("Index :", ex["index"])
    print("Rule  :", ex["rule_used"])
    print("Before:", ex["before_sentence"])
    print("After :", ex["after_sentence"])
    print("Label :", ex["error_label"])
    print("-" * 100)

if injected < TARGET_COUNT:
    print("\nWARNING:")
    print("Could not reach target count.")
    print("Injected:", injected)
    print("Target  :", TARGET_COUNT)
    print("You may need to loosen filters or add more rules.")

Syntax errors injected: 13013
Target syntax errors: 18292
Output saved to: /kaggle/working/51k_after_syntax_injection.csv
Examples saved to: /kaggle/working/syntax_error_examples_full.xlsx

Rule-wise count:
rule_used
object_predicate_inversion      5127
modifier_noun_swap              4443
temporal_phrase_displacement    3443
Name: count, dtype: int64

Label distribution:
error_label
0          27468
10          9505
10,11        281
11          1472
2           2758
           ...  
7,9,11         2
9           1401
9,10         522
9,10,11       15
9,11          97
Name: count, Length: 155, dtype: int64

First 50 examples:
Index : 34185
Rule  : modifier_noun_swap
Before: তথাপি এই ধৰ্মটো এতিয়া হাবলৰ সূত্ৰ বুলিহে জনা যায়।
After : তথাপি এই ধৰ্মটো হাবলৰ এতিয়া সূত্ৰ বুলিহে জনা যায়।
Label : 10
----------------------------------------------------------------------------------------------------
Index : 6424
Rule  : modifier_noun_swap
Before: এই গছজোপাৰ পাতবোৰ ১০-২০ ছে:মি: দীঘল আৰু ঘূৰণীয

In [6]:
import pandas as pd
import random
import re

INPUT_PATH = "/kaggle/working/51k_after_syntax_injection.csv"

OUTPUT_PATH = "/kaggle/working/51k_after_syntax_injection_completed.csv"
EXAMPLES_PATH = "/kaggle/working/syntax_error_additional_examples.xlsx"

SYNTAX_LABEL = "10"
FINAL_TARGET = 18292

random.seed(45)

df = pd.read_csv(INPUT_PATH)
df["sentence"] = df["sentence"].astype(str)
df["error_label"] = df["error_label"].astype(str)

def has_label(label_string, label):
    return label in str(label_string).split(",")

def add_label(old_label, new_label):
    old_label = str(old_label)

    if old_label == "0":
        return new_label

    labels = old_label.split(",")

    if new_label not in labels:
        labels.append(new_label)

    labels = sorted(labels, key=lambda x: int(x))
    return ",".join(labels)

def clean_spaces(sentence):
    sentence = re.sub(r"\s+", " ", sentence)
    sentence = re.sub(r"\s+([।,?!;:])", r"\1", sentence)
    return sentence.strip()

def split_punctuation(sentence):
    sentence = sentence.strip()

    if len(sentence) > 0 and sentence[-1] in "।?!":
        return sentence[:-1].strip(), sentence[-1]

    return sentence, ""

def is_safe_sentence(sentence):
    sentence = str(sentence).strip()
    words = sentence.split()

    if len(words) < 6:
        return False

    if sentence.count(",") > 2:
        return False

    if "”" in sentence or "“" in sentence or '"' in sentence:
        return False

    if "(" in sentence or ")" in sentence:
        return False

    return True

COMPOUND_LEFT = {
    "লাভ", "প্ৰদান", "যোগদান", "বহন", "স্থাপন",
    "পৰিচালনা", "প্ৰতিষ্ঠা", "প্ৰকাশ", "ব্যৱহাৰ",
    "আৰম্ভ", "সৃষ্টি", "নিৰ্মাণ", "উদ্বোধন",
    "পালন", "গ্ৰহণ", "উল্লেখ", "ৰচনা",
    "অলংকৃত", "দখল", "আৰোহণ", "সমাপ্ত",
    "সম্পন্ন", "প্ৰস্তুত", "প্ৰয়োগ"
}

AUX_RIGHT = {
    "কৰে", "কৰিছে", "কৰিছিল", "কৰিলে", "কৰা",
    "হয়", "হৈছে", "হৈছিল", "হল", "হ'ল"
}

# =========================
# OLD HIGH-QUALITY RULES
# =========================
def rule_object_predicate_inversion(sentence):
    core, punct = split_punctuation(sentence)
    words = core.split()

    candidate_positions = []

    for i in range(len(words) - 1):
        if words[i] in COMPOUND_LEFT and words[i + 1] in AUX_RIGHT:
            if i >= 2:
                candidate_positions.append(i)

    if not candidate_positions:
        return None

    i = random.choice(candidate_positions)
    new_words = words.copy()
    new_words[i], new_words[i + 1] = new_words[i + 1], new_words[i]

    return clean_spaces(" ".join(new_words) + punct)

def rule_modifier_noun_swap(sentence):
    core, punct = split_punctuation(sentence)
    words = core.split()

    modifier_suffixes = (
        "ীয়া", "িয়া", "ৰীয়া", "জাতীয়", "স্থানীয়",
        "সামাজিক", "ৰাজনৈতিক", "ধাৰ্মিক", "সাংস্কৃতিক",
        "ঐতিহাসিক", "বৈজ্ঞানিক", "প্ৰাকৃতিক", "ভৌগোলিক",
        "মানসিক", "শাৰীৰিক", "মূলক", "সম্পৰ্কীয়", "বিষয়ক",
        "হালধীয়া", "কলা", "বগা", "ৰঙা", "নীলা"
    )

    candidate_positions = []

    for i in range(len(words) - 1):
        if words[i].endswith(modifier_suffixes):
            if words[i + 1] in AUX_RIGHT:
                continue
            if re.search(r"[।,?!;:]", words[i + 1]):
                continue
            candidate_positions.append(i)

    if not candidate_positions:
        return None

    i = random.choice(candidate_positions)
    new_words = words.copy()
    new_words[i], new_words[i + 1] = new_words[i + 1], new_words[i]

    return clean_spaces(" ".join(new_words) + punct)

def rule_temporal_phrase_displacement(sentence):
    core, punct = split_punctuation(sentence)
    words = core.split()

    temporal_markers = (
        "চনত", "চনতে", "মাহত", "দিনা", "সময়ত",
        "পিছত", "আগতে", "কালত", "বছৰত"
    )

    candidate_positions = []

    for i, word in enumerate(words[:6]):
        if word.endswith(temporal_markers):
            candidate_positions.append(i)

    if not candidate_positions:
        return None

    i = random.choice(candidate_positions)
    start = max(0, i - 1)
    end = i + 1

    temporal_phrase = words[start:end]
    remaining = words[:start] + words[end:]

    if len(remaining) < 5:
        return None

    insert_pos = len(remaining) - 1

    if insert_pos <= 2:
        return None

    new_words = remaining[:insert_pos] + temporal_phrase + remaining[insert_pos:]

    return clean_spaces(" ".join(new_words) + punct)

# =========================
# NEW ADDITIONAL SYNTAX RULES
# =========================
def rule_hisape_phrase_shift(sentence):
    """
    X হিচাপে Y -> হিচাপে X Y
    """

    core, punct = split_punctuation(sentence)
    words = core.split()

    candidate_positions = []

    for i in range(1, len(words) - 2):
        if words[i] == "হিচাপে":
            candidate_positions.append(i)

    if not candidate_positions:
        return None

    i = random.choice(candidate_positions)

    new_words = words.copy()
    new_words[i - 1], new_words[i] = new_words[i], new_words[i - 1]

    return clean_spaces(" ".join(new_words) + punct)

def rule_genitive_phrase_shift(sentence):
    """
    Xৰ Y -> Y Xৰ
    Example: অসমৰ ইতিহাস -> ইতিহাস অসমৰ
    """

    core, punct = split_punctuation(sentence)
    words = core.split()

    candidate_positions = []

    for i in range(len(words) - 1):
        if words[i].endswith("ৰ") and len(words[i]) > 2:
            if words[i + 1] in AUX_RIGHT:
                continue
            candidate_positions.append(i)

    if not candidate_positions:
        return None

    i = random.choice(candidate_positions)

    new_words = words.copy()
    new_words[i], new_words[i + 1] = new_words[i + 1], new_words[i]

    return clean_spaces(" ".join(new_words) + punct)

def rule_quantifier_noun_shift(sentence):
    """
    বহু/কিছু/কেইবাখন + noun -> noun + বহু/কিছু/কেইবাখন
    """

    core, punct = split_punctuation(sentence)
    words = core.split()

    quantifiers = {
        "বহু", "কিছু", "কেইবাখন", "কেইজন", "কেইটা",
        "অনেক", "সকলো", "বিভিন্ন", "প্ৰায়"
    }

    candidate_positions = []

    for i in range(len(words) - 1):
        if words[i] in quantifiers:
            if words[i + 1] in AUX_RIGHT:
                continue
            candidate_positions.append(i)

    if not candidate_positions:
        return None

    i = random.choice(candidate_positions)

    new_words = words.copy()
    new_words[i], new_words[i + 1] = new_words[i + 1], new_words[i]

    return clean_spaces(" ".join(new_words) + punct)

def rule_location_phrase_late_shift(sentence):
    """
    Moves a locative phrase slightly later without changing bibhakti.
    Does not remove or replace ত/লৈ.
    """

    core, punct = split_punctuation(sentence)
    words = core.split()

    candidate_positions = []

    for i in range(1, min(len(words) - 3, 8)):
        if words[i].endswith("ত") or words[i].endswith("লৈ"):
            candidate_positions.append(i)

    if not candidate_positions:
        return None

    i = random.choice(candidate_positions)

    phrase = [words[i]]
    remaining = words[:i] + words[i + 1:]

    insert_pos = min(i + 2, len(remaining) - 1)

    if insert_pos <= i:
        return None

    new_words = remaining[:insert_pos] + phrase + remaining[insert_pos:]

    return clean_spaces(" ".join(new_words) + punct)

def inject_syntax_error(sentence):
    if not is_safe_sentence(sentence):
        return None, None

    rules = [
        ("object_predicate_inversion", rule_object_predicate_inversion),
        ("modifier_noun_swap", rule_modifier_noun_swap),
        ("temporal_phrase_displacement", rule_temporal_phrase_displacement),
        ("hisape_phrase_shift", rule_hisape_phrase_shift),
        ("genitive_phrase_shift", rule_genitive_phrase_shift),
        ("quantifier_noun_shift", rule_quantifier_noun_shift),
        ("location_phrase_late_shift", rule_location_phrase_late_shift)
    ]

    random.shuffle(rules)

    for rule_name, rule_func in rules:
        new_sentence = rule_func(sentence)

        if new_sentence and new_sentence != sentence:
            return new_sentence, rule_name

    return None, None

# =========================
# COUNT CURRENT SYNTAX
# =========================
current_syntax_count = df["error_label"].apply(lambda x: has_label(x, SYNTAX_LABEL)).sum()
remaining_target = FINAL_TARGET - current_syntax_count

print("Current syntax count:", current_syntax_count)
print("Final target:", FINAL_TARGET)
print("Need to add:", remaining_target)

# =========================
# ADD ONLY TO ROWS WITHOUT SYNTAX LABEL
# =========================
candidate_indices = df.index[
    (~df["error_label"].apply(lambda x: has_label(x, SYNTAX_LABEL))) &
    (df["sentence"].apply(is_safe_sentence))
].tolist()

random.shuffle(candidate_indices)

injected = 0
examples = []

for idx in candidate_indices:
    if injected >= remaining_target:
        break

    original_sentence = df.at[idx, "sentence"]
    new_sentence, rule_name = inject_syntax_error(original_sentence)

    if new_sentence is None:
        continue

    df.at[idx, "sentence"] = new_sentence
    df.at[idx, "error_label"] = add_label(df.at[idx, "error_label"], SYNTAX_LABEL)

    examples.append({
        "index": idx,
        "rule_used": rule_name,
        "before_sentence": original_sentence,
        "after_sentence": new_sentence,
        "error_label": df.at[idx, "error_label"]
    })

    injected += 1

# =========================
# SAVE
# =========================
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

examples_df = pd.DataFrame(examples)
examples_df.to_excel(EXAMPLES_PATH, index=False)

final_syntax_count = df["error_label"].apply(lambda x: has_label(x, SYNTAX_LABEL)).sum()

print("\nAdditional syntax errors injected:", injected)
print("Final syntax count:", final_syntax_count)
print("Final target:", FINAL_TARGET)
print("Output saved to:", OUTPUT_PATH)
print("Examples saved to:", EXAMPLES_PATH)

print("\nRule-wise count for additional injections:")
print(examples_df["rule_used"].value_counts())

print("\nFirst 50 additional examples:")
for ex in examples[:50]:
    print("Index :", ex["index"])
    print("Rule  :", ex["rule_used"])
    print("Before:", ex["before_sentence"])
    print("After :", ex["after_sentence"])
    print("Label :", ex["error_label"])
    print("-" * 100)

if final_syntax_count < FINAL_TARGET:
    print("\nWARNING:")
    print("Still below target.")
    print("Final syntax count:", final_syntax_count)
    print("Target:", FINAL_TARGET)
else:
    print("\nTarget reached.")

Current syntax count: 13013
Final target: 18292
Need to add: 5279

Additional syntax errors injected: 5279
Final syntax count: 18292
Final target: 18292
Output saved to: /kaggle/working/51k_after_syntax_injection_completed.csv
Examples saved to: /kaggle/working/syntax_error_additional_examples.xlsx

Rule-wise count for additional injections:
rule_used
genitive_phrase_shift           3460
location_phrase_late_shift      1392
quantifier_noun_shift            246
hisape_phrase_shift              100
temporal_phrase_displacement      39
object_predicate_inversion        28
modifier_noun_swap                14
Name: count, dtype: int64

First 50 additional examples:
Index : 35592
Rule  : location_phrase_late_shift
Before: কিন্তু আলোচনা কৰিবলৈ যোৱা দলটোক তেওঁলোকে হত্যা কৰে।
After : কিন্তু আলোচনা যোৱা দলটোক কৰিবলৈ তেওঁলোকে হত্যা কৰে।
Label : 10
----------------------------------------------------------------------------------------------------
Index : 50716
Rule  : genitive_phrase_shift
Befor

In [1]:
import pandas as pd
import re

# =========================
# PATHS
# =========================
DATASET_PATH = "/kaggle/input/datasets/shruteesaha/51k-after-syntax-injection/51k_after_syntax_injection.csv"

BIBHAKTI_RULES_PATH = "/kaggle/input/datasets/shruteesaha/bibhakti-error-rules/bibhakti_error_rules.xlsx"

OUTPUT_PATH = "/kaggle/working/bibhakti_word_frequencies.csv"

# =========================
# LOAD FILES
# =========================
df = pd.read_csv(DATASET_PATH)
rules_df = pd.read_excel(BIBHAKTI_RULES_PATH)

# =========================
# CLEAN COLUMN NAMES
# =========================
rules_df.columns = rules_df.columns.str.strip()

print("Rule columns:", rules_df.columns.tolist())

# =========================
# GET CORRECT WORDS
# =========================
correct_col = "correct_word"

if correct_col not in rules_df.columns:
    raise ValueError(f"Column '{correct_col}' not found. Available columns: {rules_df.columns.tolist()}")

correct_words = (
    rules_df[correct_col]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

# =========================
# COUNT FREQUENCY IN DATASET
# =========================
sentences = df["sentence"].astype(str)

results = []

for word in correct_words:
    pattern = r'(?<!\S)' + re.escape(word) + r'(?!\S)'

    freq = sentences.str.count(pattern).sum()

    results.append({
        "correct_word": word,
        "frequency": int(freq)
    })

freq_df = pd.DataFrame(results)

freq_df = freq_df.sort_values(
    by="frequency",
    ascending=False
).reset_index(drop=True)

# =========================
# DISPLAY FULL RESULT
# =========================
print(freq_df.to_string(index=False))

# =========================
# SAVE OUTPUT
# =========================
freq_df.to_csv(OUTPUT_PATH, index=False)

print(f"\nSaved: {OUTPUT_PATH}")

Rule columns: ['correct_word', 'incorrect_word']
       correct_word  frequency
              সময়ত       1024
               অসমৰ        708
              ভাৰতৰ        700
             অঞ্চলত        393
             মানুহৰ        391
              ভাষাৰ        357
    বিশ্ববিদ্যালয়ৰ        341
              জীৱনৰ        328
              ভাষাত        303
              সময়ৰ        302
              জিলাৰ        300
               দেশৰ        284
             অঞ্চলৰ        249
               অসমত        246
           পৰিয়ালৰ        234
             চৰকাৰৰ        228
             ৰাজ্যৰ        223
               বছৰৰ        221
                দলৰ        217
              ভাৰতত        198
             বিভাগৰ        197
           বিজ্ঞানৰ        185
               দিনৰ        172
            শিক্ষাৰ        169
              সমাজৰ        166
              সমাজত        163
               নদীৰ        162
             সংগীতৰ        161
               কামত        157
              ধৰ্মৰ  

In [4]:
import pandas as pd
import re
from collections import Counter

# =========================
# PATH
# =========================
DATASET_PATH = "/kaggle/input/datasets/shruteesaha/51k-after-syntax-injection/51k_after_syntax_injection.csv"

# =========================
# LOAD
# =========================
df = pd.read_csv(DATASET_PATH)

# =========================
# TOKENIZE
# =========================
all_words = []

for sentence in df["sentence"].astype(str):

    sentence = re.sub(r'[।,;:!?()"\'\[\]{}]', ' ', sentence)

    words = sentence.split()

    all_words.extend(words)

# =========================}
# COUNT
# =========================
word_freq = Counter(all_words)

# =========================
# TOP 100
# =========================
top_100 = pd.DataFrame(
    word_freq.most_common(300),
    columns=["word", "frequency"]
)

print(top_100.to_string(index=False))

# =========================
# SAVE
# =========================
top_100.to_csv(
    "/kaggle/working/top_100_words.csv",
    index=False
)

print("\nSaved: /kaggle/working/top_100_words.csv")

           word  frequency
            আৰু      17319
            হয়       8512
             এই       8230
            কৰে       6957
            কৰা       6766
           তেওঁ       5953
            পৰা       4661
           বাবে       4610
            চনত       4220
          হৈছিল       3895
         কৰিছিল       3762
             বা       3286
           হৈছে       3048
         কৰিবলৈ       2938
           আছিল       2928
             হৈ       2864
             এক       2823
          তেওঁৰ       2727
         হিচাপে       2691
            এটা       2664
           বুলি       2470
          ইয়াৰ       2407
            আছে       2401
            লাভ       2382
            কৰি       2314
           যায়       2249
            চনৰ       1967
              হ       1869
        বিভিন্ন       1848
            থকা       1741
           সৈতে       1620
             যে       1620
            সেই       1603
           কোনো       1576
            লগত       1565
           কৰাৰ       1530
 

In [8]:
import pandas as pd
import re

# =========================
# PATHS
# =========================
DATASET_PATH = "/kaggle/input/datasets/shruteesaha/51k-after-syntax-injection/51k_after_syntax_injection.csv"

BIBHAKTI_RULES_PATH = "/kaggle/input/datasets/shruteesaha/bibhakti-whatever-final/bibhakti_rules_final_cleaned_8k_no_karak_overlap_final_final.xlsx"

OUTPUT_PATH = "/kaggle/working/bibhakti_word_frequencies1.csv"

# =========================
# LOAD FILES
# =========================
df = pd.read_csv(DATASET_PATH)
rules_df = pd.read_excel(BIBHAKTI_RULES_PATH)

# =========================
# CLEAN COLUMN NAMES
# =========================
rules_df.columns = rules_df.columns.str.strip()

print("Rule columns:", rules_df.columns.tolist())

# =========================
# GET CORRECT WORDS
# =========================
correct_col = "correct_word"

if correct_col not in rules_df.columns:
    raise ValueError(f"Column '{correct_col}' not found. Available columns: {rules_df.columns.tolist()}")

correct_words = (
    rules_df[correct_col]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

# =========================
# COUNT FREQUENCY IN DATASET
# =========================
sentences = df["sentence"].astype(str)

results = []

for word in correct_words:
    pattern = r'(?<!\S)' + re.escape(word) + r'(?!\S)'

    freq = sentences.str.count(pattern).sum()

    results.append({
        "correct_word": word,
        "frequency": int(freq)
    })

freq_df = pd.DataFrame(results)

freq_df = freq_df.sort_values(
    by="frequency",
    ascending=False
).reset_index(drop=True)

# =========================
# DISPLAY FULL RESULT
# =========================
print(freq_df.to_string(index=False))

# =========================
# SAVE OUTPUT
# =========================
freq_df.to_csv(OUTPUT_PATH, index=False)

print(f"\nSaved: {OUTPUT_PATH}")

Rule columns: ['correct_word', 'incorrect_word']
   correct_word  frequency
           অসমৰ        708
          ভাৰতৰ        700
         মানুহৰ        391
          ভাষাৰ        357
বিশ্ববিদ্যালয়ৰ        341
          জীৱনৰ        328
          সময়ৰ        302
          জিলাৰ        300
           দেশৰ        284
         অঞ্চলৰ        249
       পৰিয়ালৰ        234
         চৰকাৰৰ        228
         ৰাজ্যৰ        223
           বছৰৰ        221
            দলৰ        217
         বিভাগৰ        197
           ছবিৰ        197
           লোকৰ        196
       বিজ্ঞানৰ        185
           মাহৰ        182
          শব্দৰ        175
           দিনৰ        172
        শিক্ষাৰ        169
           ৰজাৰ        168
          সমাজৰ        166
           নদীৰ        162
           ৰোগৰ        161
         সংগীতৰ        161
          ধৰ্মৰ        157
            ঘৰৰ        150
       সাহিত্যৰ        149
           গীতৰ        142
        অভিনয়ৰ        140
     চলচ্চিত্ৰৰ        139
      

In [1]:
import pandas as pd
import random
import re
from collections import defaultdict

# =========================
# PATHS
# =========================
INPUT_PATH = "/kaggle/input/datasets/shruteesaha/51k-after-syntax-injection/51k_after_syntax_injection.csv"

BIBHAKTI_RULES_PATH = "/kaggle/input/datasets/shruteesaha/bibhakti-whatever-final/bibhakti_rules_final_cleaned_8k_no_karak_overlap_final_final.xlsx"

OUTPUT_PATH = "/kaggle/working/51k_after_bibhakti_injection.csv"
REPORT_PATH = "/kaggle/working/bibhakti_injection_report.csv"

BIBHAKTI_LABEL = "8"   # change if your actual Bibhakti label is different
TARGET_COUNT = 7740   # 52/343 of 51k approx

random.seed(45)

# =========================
# LOAD
# =========================
df = pd.read_csv(INPUT_PATH)
rules_df = pd.read_excel(BIBHAKTI_RULES_PATH)

df["sentence"] = df["sentence"].astype(str)
df["error_label"] = df["error_label"].astype(str)

rules_df.columns = rules_df.columns.str.strip()

print("Rule columns:", rules_df.columns.tolist())

required_cols = {"correct_word", "incorrect_word"}
missing = required_cols - set(rules_df.columns)

if missing:
    raise ValueError(f"Missing required columns: {missing}")

rules_df = rules_df[["correct_word", "incorrect_word"]].dropna()

rules_df["correct_word"] = rules_df["correct_word"].astype(str).str.strip()
rules_df["incorrect_word"] = rules_df["incorrect_word"].astype(str).str.strip()

rules_df = rules_df[
    (rules_df["correct_word"] != "") &
    (rules_df["incorrect_word"] != "") &
    (rules_df["correct_word"] != rules_df["incorrect_word"])
].drop_duplicates().reset_index(drop=True)

print("Total Bibhakti rules:", len(rules_df))

# =========================
# HELPERS
# =========================
def add_label(old_label, new_label):
    old_label = str(old_label).strip()

    if old_label == "0":
        return new_label

    labels = [x.strip() for x in old_label.split(",") if x.strip()]

    if new_label not in labels:
        labels.append(new_label)

    labels = sorted(labels, key=lambda x: int(x) if x.isdigit() else 999)
    return ",".join(labels)


def whole_word_pattern(word):
    return r"(?<!\S)" + re.escape(word) + r"(?!\S)"


def replace_once(sentence, correct_word, incorrect_word):
    pattern = whole_word_pattern(correct_word)
    return re.sub(pattern, incorrect_word, sentence, count=1)


def is_safe_sentence(sentence):
    sentence = str(sentence).strip()

    if len(sentence.split()) < 4:
        return False

    # Avoid very messy lines
    if sentence.count("।") > 1:
        return False

    if "http" in sentence.lower() or "www" in sentence.lower():
        return False

    return True


# =========================
# BUILD MATCH CANDIDATES
# =========================
sentences = df["sentence"].astype(str)

rule_to_indices = defaultdict(list)

for rule_idx, row in rules_df.iterrows():
    correct_word = row["correct_word"]
    pattern = whole_word_pattern(correct_word)

    matched_indices = df.index[
        sentences.str.contains(pattern, regex=True, na=False)
    ].tolist()

    matched_indices = [
        idx for idx in matched_indices
        if is_safe_sentence(df.at[idx, "sentence"])
    ]

    if matched_indices:
        rule_to_indices[rule_idx] = matched_indices

total_possible_matches = sum(len(v) for v in rule_to_indices.values())

print("Total possible Bibhakti matches:", total_possible_matches)

usable_rules = len(rule_to_indices)
print("Usable rules:", usable_rules)

if total_possible_matches < TARGET_COUNT:
    print("\nWARNING:")
    print("Possible matches are less than target.")
    print("Possible:", total_possible_matches)
    print("Target  :", TARGET_COUNT)

# =========================
# CREATE CANDIDATE POOL
# =========================
candidate_pool = []

for rule_idx, indices in rule_to_indices.items():
    row = rules_df.loc[rule_idx]

    for idx in indices:
        candidate_pool.append({
            "row_idx": idx,
            "rule_idx": rule_idx,
            "correct_word": row["correct_word"],
            "incorrect_word": row["incorrect_word"]
        })

random.shuffle(candidate_pool)

# =========================
# INJECT
# =========================
used_rows = set()
injection_records = []

for cand in candidate_pool:
    if len(injection_records) >= TARGET_COUNT:
        break

    idx = cand["row_idx"]

    # one Bibhakti injection per row
    if idx in used_rows:
        continue

    original_sentence = df.at[idx, "sentence"]

    pattern = whole_word_pattern(cand["correct_word"])

    if not re.search(pattern, original_sentence):
        continue

    new_sentence = replace_once(
        original_sentence,
        cand["correct_word"],
        cand["incorrect_word"]
    )

    if new_sentence == original_sentence:
        continue

    old_label = str(df.at[idx, "error_label"])
    new_label = add_label(old_label, BIBHAKTI_LABEL)

    df.at[idx, "sentence"] = new_sentence
    df.at[idx, "error_label"] = new_label

    used_rows.add(idx)

    injection_records.append({
        "row_idx": idx,
        "correct_word": cand["correct_word"],
        "incorrect_word": cand["incorrect_word"],
        "before_sentence": original_sentence,
        "after_sentence": new_sentence,
        "old_label": old_label,
        "new_label": new_label
    })

# =========================
# SAVE
# =========================
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

report_df = pd.DataFrame(injection_records)
report_df.to_csv(REPORT_PATH, index=False, encoding="utf-8-sig")

print("\nBibhakti errors injected:", len(injection_records))
print("Target Bibhakti errors:", TARGET_COUNT)

print("\nSaved injected dataset:")
print(OUTPUT_PATH)

print("\nSaved Bibhakti injection report:")
print(REPORT_PATH)

print("\nTop injected correct words:")
print(report_df["correct_word"].value_counts().head(30))

print("\nLabel distribution after Bibhakti injection:")
print(df["error_label"].value_counts().sort_index())

print("\nExamples:")
pd.set_option("display.max_colwidth", None)
print(
    report_df[
        ["correct_word", "incorrect_word", "before_sentence", "after_sentence", "new_label"]
    ].head(30).to_string(index=False)
)

if len(injection_records) < TARGET_COUNT:
    print("\nWARNING:")
    print("Could not reach target count.")
    print("Injected:", len(injection_records))
    print("Target  :", TARGET_COUNT)
    print("You may need to increase the rule sheet or loosen sentence filters.")

Rule columns: ['correct_word', 'incorrect_word']
Total Bibhakti rules: 273
Total possible Bibhakti matches: 14230
Usable rules: 271

Bibhakti errors injected: 7740
Target Bibhakti errors: 7740

Saved injected dataset:
/kaggle/working/51k_after_bibhakti_injection.csv

Saved Bibhakti injection report:
/kaggle/working/bibhakti_injection_report.csv

Top injected correct words:
correct_word
অসমৰ               373
ভাৰতৰ              365
মানুহৰ             218
বিশ্ববিদ্যালয়ৰ    179
ভাষাৰ              175
জীৱনৰ              170
দেশৰ               143
সময়ৰ              142
জিলাৰ              140
অঞ্চলৰ             127
চৰকাৰৰ             123
পৰিয়ালৰ           122
বছৰৰ               116
ছবিৰ               116
ৰাজ্যৰ             109
বিভাগৰ             108
ৰজাৰ               101
লোকৰ               100
দলৰ                 96
মাহৰ                95
সংগীতৰ              94
দিনৰ                94
শব্দৰ               94
বিজ্ঞানৰ            91
ধৰ্মৰ               86
সমাজৰ               85
শিক্ষাৰ      

In [1]:
import pandas as pd
import random
import re
from collections import defaultdict

INPUT_PATH = "/kaggle/input/datasets/shruteesaha/51k-after-bibhakti-injection/51k_after_bibhakti_injection.csv"
RULES_PATH = "/kaggle/input/datasets/shruteesaha/assamese-non-dictionary-spelling/assamese_non_dictionary_spelling_pattern_rules_revised.xlsx"
OUTPUT_PATH = "/kaggle/working/51k_after_non_dictionary_spelling_injection.csv"
EXAMPLES_PATH = "/kaggle/working/non_dictionary_spelling_error_examples.xlsx"

SPELLING_LABEL = "1"
random.seed(45)

df = pd.read_csv(INPUT_PATH)
rules_df = pd.read_excel(RULES_PATH)

df["sentence"] = df["sentence"].astype(str)
df["error_label"] = df["error_label"].astype(str)

rules_df["incorrect"] = rules_df["incorrect"].astype(str)
rules_df["correct"] = rules_df["correct"].astype(str)

rules = list(rules_df[["incorrect", "correct"]].itertuples(index=False, name=None))

TARGET_TOTAL = round(len(df) * 185 / 343)

single_count = round(TARGET_TOTAL * 174 / 195)
double_count = round(TARGET_TOTAL * 18 / 195)
triple_count = round(TARGET_TOTAL * 3 / 195)

while single_count + double_count + triple_count > TARGET_TOTAL:
    single_count -= 1

while single_count + double_count + triple_count < TARGET_TOTAL:
    single_count += 1

target_edit_plan = ([1] * single_count) + ([2] * double_count) + ([3] * triple_count)
random.shuffle(target_edit_plan)

def add_label(old_label, new_label):
    old_label = str(old_label)
    if old_label == "0":
        return new_label
    labels = old_label.split(",")
    if new_label not in labels:
        labels.append(new_label)
    return ",".join(labels)

def get_words(sentence):
    return re.findall(r"[\u0980-\u09FF]+", sentence)

def apply_one_spelling_error(sentence, used_spans):
    possible = []

    for incorrect, correct in rules:
        if not correct or correct == "nan":
            continue

        for match in re.finditer(re.escape(correct), sentence):
            start, end = match.span()

            overlap = False
            for u_start, u_end in used_spans:
                if not (end <= u_start or start >= u_end):
                    overlap = True
                    break

            if not overlap:
                possible.append((start, end, correct, incorrect))

    if not possible:
        return sentence, None

    start, end, correct, incorrect = random.choice(possible)
    new_sentence = sentence[:start] + incorrect + sentence[end:]
    used_spans.append((start, start + len(incorrect)))

    return new_sentence, {
        "correct_pattern": correct,
        "incorrect_pattern": incorrect
    }

examples = []
available_indices = list(df.index)
random.shuffle(available_indices)

success = 0
plan_idx = 0

for idx in available_indices:
    if plan_idx >= len(target_edit_plan):
        break

    required_edits = target_edit_plan[plan_idx]
    old_sentence = df.at[idx, "sentence"]
    new_sentence = old_sentence
    used_spans = []
    applied = []

    for _ in range(required_edits):
        new_sentence, info = apply_one_spelling_error(new_sentence, used_spans)
        if info is None:
            break
        applied.append(info)

    if len(applied) == required_edits and new_sentence != old_sentence:
        df.at[idx, "sentence"] = new_sentence
        df.at[idx, "error_label"] = add_label(df.at[idx, "error_label"], SPELLING_LABEL)

        examples.append({
            "row_index": idx,
            "number_of_spelling_errors": required_edits,
            "before_sentence": old_sentence,
            "after_sentence": new_sentence,
            "applied_rules": str(applied),
            "new_label": df.at[idx, "error_label"]
        })

        success += 1
        plan_idx += 1

df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
pd.DataFrame(examples).to_excel(EXAMPLES_PATH, index=False)

print("Target non-dictionary spelling sentences:", TARGET_TOTAL)
print("Injected:", success)
print("Single-error sentences:", sum(1 for e in examples if e["number_of_spelling_errors"] == 1))
print("Double-error sentences:", sum(1 for e in examples if e["number_of_spelling_errors"] == 2))
print("Triple-error sentences:", sum(1 for e in examples if e["number_of_spelling_errors"] == 3))
print("Output saved to:", OUTPUT_PATH)
print("Examples saved to:", EXAMPLES_PATH)

Target non-dictionary spelling sentences: 27507
Injected: 27507
Single-error sentences: 24545
Double-error sentences: 2539
Triple-error sentences: 423
Output saved to: /kaggle/working/51k_after_non_dictionary_spelling_injection.csv
Examples saved to: /kaggle/working/non_dictionary_spelling_error_examples.xlsx


In [6]:
INPUT_PATH = "/kaggle/working/non_dictionary_spelling_error_examples.xlsx"

df_spell = pd.read_excel(INPUT_PATH)
df_spell.head(50)

,row_index,number_of_spelling_errors,before_sentence,after_sentence,applied_rules,new_label
0,810,1,"ভাগৱত পুৰাণৰ উপৰিও সমসাময়িক ভাৰতীয় লোক নাটক,...","ভাগৱত পুৰাণৰ উপৰিও সমসাময়িক ভাৰতীয় লোক নাটক,...","[{'correct_pattern': 'ৰ', 'incorrect_pattern':...","2,1"
1,24276,1,"২৬ বছৰৰ পিছত ৭ জুন, ২০১০ চনত আদালতে ৰায় পৰ্যন...","২৬ বছৰৰ পিছত ৭ জুন, ২০১০ চনত আদালতৈ ৰায় পৰ্যন...","[{'correct_pattern': 'ে', 'incorrect_pattern':...","10,1"
2,798,1,ছিটাৰ ব্যৱহাৰক ফলত এইৰোগৰ প্ৰাদূৰ্ভাৱ কমিব বুল...,ছিটাৰ ব্যৱহাৰক ফলৎ এইৰোগৰ প্ৰাদূৰ্ভাৱ কমিব বুল...,"[{'correct_pattern': 'ত', 'incorrect_pattern':...","8,1"
3,50094,1,এটা সাক্ষাৎকাৰত কণ্ঠশিল্পী নীলিমা খাতুনে সৰুৰে...,এটা সাক্ষাৎকাৰত কণ্ঠশিল্পী নীলিমা খাৎুনে সৰুৰে...,"[{'correct_pattern': 'ত', 'incorrect_pattern':...",1
4,3370,2,"মহাৰজত জয়ন্তী উপলক্ষে প্ৰকাশিত মুখপত্ৰৰ নাম ""...","মহাৰজত জয়ন্ৎী উপলক্ষে প্ৰকাষিত মুখপত্ৰৰ নাম ""...","[{'correct_pattern': 'ত', 'incorrect_pattern':...",1
5,33570,1,নীতিশিক্ষা নথকা এনে সাধুবোৰ শিশুক কাৰণে অনুপযু...,নীতিশিক্ষা নথকা এনে সাধুবোৰ শিশুক কাৰণে অনুপযু...,"[{'correct_pattern': 'ব', 'incorrect_pattern':...","8,1"
6,1721,1,দুৰ্গা পূজা হৈছে শক্তিৰ অধিষ্ঠাত্ৰী দুৰ্গা দেৱ...,দুৰ্গা পূজা হৈছে শক্তিৰ অধিষাত্ৰী দুৰ্গা দেৱীৰ...,"[{'correct_pattern': 'ষ্ঠ', 'incorrect_pattern...",1
7,1248,1,"Hugh Hughes,এ তেওঁৰ 'ইণ্ডিয়ান লোকোমোটিভ' গ্ৰন...","Hugh Hughes,এ তেওঁৰ 'ইণ্ডিয়ান লোকোমোটিভ' গ্ৰন...","[{'correct_pattern': 'ে', 'incorrect_pattern':...","8,10,11,1"
8,7042,1,ৰাষ্ট্ৰপতি ৰূপে কাৰ্যৰত হৈ থাকিবৰ সংসদীয় বাবে...,ৰাষ্ট্ৰপতি ৰূপে কাৰ্যৰত হৈ থাকিবৰ সংসদীয় বাবে...,"[{'correct_pattern': 'ৰ', 'incorrect_pattern':...",1
9,6598,1,কৌৰৱকে ধৰিবলৈ আন অংশগ্ৰহণকাৰীসকলে প্ৰতিযোগিতা ...,কৌৰৱকে ধৰিবলৈ আন অংশগ্ৰহণকাঢ়ীসকলে প্ৰতিযোগিতা...,"[{'correct_pattern': 'ৰ', 'incorrect_pattern':...","2,1"


In [8]:
import pandas as pd

INPUT_PATH = "/kaggle/working/51k_after_non_dictionary_spelling_injection.csv"
OUTPUT_PATH = "/kaggle/working/label_1_examples_all_combinations.xlsx"

df = pd.read_csv(INPUT_PATH)

df["error_label"] = df["error_label"].astype(str)
df["sentence"] = df["sentence"].astype(str)

label1_df = df[
    df["error_label"].apply(lambda x: "1" in [i.strip() for i in x.split(",")])
].copy()

label1_df["label_tuple"] = label1_df["error_label"].apply(
    lambda x: tuple(sorted([i.strip() for i in x.split(",")], key=int))
)

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    for label_combo, group in label1_df.groupby("label_tuple"):
        sheet_name = ",".join(label_combo)
        group[["sentence", "error_label"]].to_excel(
            writer,
            sheet_name=sheet_name[:31],
            index=False
        )

print("Done")
print("Total label 1 sentences:", len(label1_df))
print("Saved to:", OUTPUT_PATH)

print("\nCombinations containing label 1:")
print(label1_df["error_label"].value_counts().sort_index())


Done
Total label 1 sentences: 27507
Saved to: /kaggle/working/label_1_examples_all_combinations.xlsx

Combinations containing label 1:
error_label
1            12644
10,1          4250
10,11,1        126
11,1           707
2,1           1267
             ...  
8,9,11,1        10
9,1            614
9,10,1         224
9,10,11,1        7
9,11,1          42
Name: count, Length: 207, dtype: int64


In [18]:
spell_dict = "/kaggle/working/51k_after_non_dictionary_spelling_injection.csv"
output_spell_eg = pd.read_csv(spell_dict)
print(output_spell_eg.head(100).to_string(index=False))

                                                                                                                                                                                                                                                 sentence error_label
                                                                                                                                                               এসময়ত চালচিত্ৰৰ ব্যৱহাঢ় হ্ৰাস হৈছিল, কিন্তু ইয়াৰ এতিয়া যথেষ্ট জনপ্ৰিয়তা বৃদ্ধি পাইছে।        10,1
                                                                                                                                                                                                       তেওঁ ইন্দ্ৰৰ দৰেই ঐৰাৱতক বাহন হিচাপে লৈ ভ্ৰমণ কৰে।           0
                                                                                                                        গাড়ীৰ পোহৰে চকু চাঁট মাৰি ধৰা বা কোঠাৰ বিজুলী বাতি নুমাই শুই থকা অৱস্থাত বাহিৰৰ পৰা পোহৰ আহিব

In [19]:
import pandas as pd
import re

INPUT_PATH = "/kaggle/working/51k_after_non_dictionary_spelling_injection.csv"
RULE_PATH = "/kaggle/input/datasets/shruteesaha/dictionary-spelling-rules/dictionary_spelling_rules_semi.xlsx"

df = pd.read_csv(INPUT_PATH)
rules = pd.read_excel(RULE_PATH)

df["sentence"] = df["sentence"].astype(str)
df["error_label"] = df["error_label"].astype(str)

rules["correct"] = rules["correct"].astype(str).str.strip()
rules["incorrect"] = rules["incorrect"].astype(str).str.strip()

rules = rules[
    (rules["correct"] != "") &
    (rules["incorrect"] != "") &
    (rules["correct"] != rules["incorrect"])
].copy()

TARGET_COUNT = round(len(df) * (10 / 343))

def has_whole_word(sentence, word):
    pattern = rf'(?<!\S){re.escape(word)}(?!\S)'
    return bool(re.search(pattern, sentence))

rule_stats = []

matched_rows = set()

for _, rule in rules.iterrows():
    correct = rule["correct"]
    incorrect = rule["incorrect"]

    row_indices = []

    for idx, sentence in df["sentence"].items():
        if has_whole_word(sentence, correct):
            row_indices.append(idx)
            matched_rows.add(idx)

    rule_stats.append({
        "correct": correct,
        "incorrect": incorrect,
        "sentence_match_count": len(row_indices)
    })

stats_df = pd.DataFrame(rule_stats)
stats_df = stats_df.sort_values("sentence_match_count", ascending=False)

total_possible_sentences = len(matched_rows)

print("Target:", TARGET_COUNT)
print("Total possible unique sentences:", total_possible_sentences)
print("Can hit target:", total_possible_sentences >= TARGET_COUNT)
print("Shortage:", max(0, TARGET_COUNT - total_possible_sentences))

print("\nTop matching rules:")
print(stats_df.head(50).to_string(index=False))

print("\nZero-frequency rules:")
print(stats_df[stats_df["sentence_match_count"] == 0].to_string(index=False))

stats_df.to_excel("/kaggle/working/dictionary_rule_frequency_check.xlsx", index=False)
print("\nSaved:", "/kaggle/working/dictionary_rule_frequency_check.xlsx")

Target: 1487
Total possible unique sentences: 11838
Can hit target: True
Shortage: 0

Top matching rules:
correct incorrect  sentence_match_count
    কৰা      কঢ়া                  6010
    পৰা      পঢ়া                  4239
  গ্ৰহণ     গ্ৰহন                   453
    দিন       দীণ                   353
    দেশ     দ্বেষ                   312
   পানী      পাণি                   219
 লক্ষ্য      লক্ষ                   139
   পঢ়া       পৰা                   107
   কঢ়া       কৰা                   105
   কোটি      কুটি                    97
    গুণ       গুন                    87
    পাত       পাট                    86
    গাত      গাঁত                    69
   বাঁহ       বাহ                    64
    সুৰ       সূৰ                    57
    পোন       পোণ                    54
     বন        বণ                    50
    বাট       বাত                    44
    বাহ      বাঁহ                    43
    কটা      কঁটা                    42
    কোন       কোণ                    40
    সাপ       

In [2]:
import pandas as pd
import random
import re

INPUT_PATH = "/kaggle/input/datasets/shruteesaha/51k-after-non-dictionary-spelling-injection/51k_after_non_dictionary_spelling_injection (1).csv"
RULE_PATH = "/kaggle/input/datasets/shruteesaha/dictionary-spelling-rules/dictionary_spelling_rules_semi.xlsx"

OUTPUT_PATH = "/kaggle/working/51k_after_dictionary_spelling_injection.csv"
EXAMPLES_PATH = "/kaggle/working/dictionary_spelling_error_examples.xlsx"

NEW_LABEL = "1D"
TARGET_COUNT = round(51000 * (10 / 343))

random.seed(45)

BLOCKED_PAIRS = {
    ("কৰা", "কঢ়া"),
    ("কঢ়া", "কৰা"),
    ("পৰা", "পঢ়া"),
    ("পঢ়া", "পৰা")
}

df = pd.read_csv(INPUT_PATH)
rules = pd.read_excel(RULE_PATH)

df["sentence"] = df["sentence"].astype(str)
df["error_label"] = df["error_label"].astype(str)

rules["correct"] = rules["correct"].astype(str).str.strip()
rules["incorrect"] = rules["incorrect"].astype(str).str.strip()

rules = rules[
    (rules["correct"] != "") &
    (rules["incorrect"] != "") &
    (rules["correct"] != rules["incorrect"])
].drop_duplicates(subset=["correct", "incorrect"]).copy()

rules = rules[
    ~rules.apply(lambda x: (x["correct"], x["incorrect"]) in BLOCKED_PAIRS, axis=1)
].copy()

rule_map = {}

for _, row in rules.iterrows():
    rule_map.setdefault(row["correct"], []).append(row["incorrect"])

correct_words = sorted(rule_map.keys(), key=len, reverse=True)

combined_pattern = re.compile(
    r'(?<!\S)(' + "|".join(re.escape(w) for w in correct_words) + r')(?!\S)'
)

def add_label(old_label, new_label):
    old_label = str(old_label)
    if old_label == "0":
        return new_label
    labels = [x.strip() for x in old_label.split(",")]
    if new_label not in labels:
        labels.append(new_label)
    return ",".join(labels)

def has_label(label, target):
    return target in [x.strip() for x in str(label).split(",")]

eligible = []

for idx, row in df.iterrows():
    if has_label(row["error_label"], NEW_LABEL):
        continue

    matches = list(combined_pattern.finditer(row["sentence"]))

    if matches:
        eligible.append(idx)

random.shuffle(eligible)

examples = []
count = 0

for idx in eligible:
    if count >= TARGET_COUNT:
        break

    before_sentence = df.at[idx, "sentence"]
    matches = list(combined_pattern.finditer(before_sentence))

    if not matches:
        continue

    match = random.choice(matches)
    correct = match.group(1)
    incorrect = random.choice(rule_map[correct])

    start, end = match.span()
    after_sentence = before_sentence[:start] + incorrect + before_sentence[end:]

    if after_sentence == before_sentence:
        continue

    df.at[idx, "sentence"] = after_sentence
    df.at[idx, "error_label"] = add_label(df.at[idx, "error_label"], NEW_LABEL)

    examples.append({
        "correct_word": correct,
        "incorrect_word": incorrect,
        "before_sentence": before_sentence,
        "after_sentence": after_sentence,
        "new_label": df.at[idx, "error_label"]
    })

    count += 1

examples_df = pd.DataFrame(examples)

df.to_csv(OUTPUT_PATH, index=False)
examples_df.to_excel(EXAMPLES_PATH, index=False)

print("Target dictionary errors:", TARGET_COUNT)
print("Injected dictionary errors:", count)
print("Output saved to:", OUTPUT_PATH)
print("Examples saved to:", EXAMPLES_PATH)
print(examples_df.head(30).to_string(index=False))

Target dictionary errors: 1487
Injected dictionary errors: 1487
Output saved to: /kaggle/working/51k_after_dictionary_spelling_injection.csv
Examples saved to: /kaggle/working/dictionary_spelling_error_examples.xlsx
correct_word incorrect_word                                                                                                                                                                                                                                        before_sentence                                                                                                                                                                                                                                         after_sentence new_label
        পানী           পাণি                                                                                                                                                 ইয়াত এটা জলপ্ঢ়পাত জলাধাৰাৰৰ সাহায্যৰে নদীক পানী জলবিদ্যুৎ উৎপাদনৰ বাবে এটা খাল

In [3]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/shruteesaha/51k-after-final-injection/51k_after_dictionary_spelling_injection (1).csv")

def count_errors(label):
    label = str(label).strip()

    if label == "0":
        return 0

    return len([x.strip() for x in label.split(",") if x.strip()])

df["num_errors"] = df["error_label"].apply(count_errors)

distribution = (
    df["num_errors"]
    .value_counts()
    .sort_index()
    .reset_index()
)

distribution.columns = ["Number_of_Errors", "Sentence_Count"]

print(distribution)

print("\nPercentage Distribution:")
for _, row in distribution.iterrows():
    pct = row["Sentence_Count"] * 100 / len(df)
    print(f"{row['Number_of_Errors']} errors : {row['Sentence_Count']} ({pct:.2f}%)")

   Number_of_Errors  Sentence_Count
0                 0           10654
1                 1           21205
2                 2           13779
3                 3            4361
4                 4             868
5                 5             119
6                 6              13
7                 7               1

Percentage Distribution:
0 errors : 10654 (20.89%)
1 errors : 21205 (41.58%)
2 errors : 13779 (27.02%)
3 errors : 4361 (8.55%)
4 errors : 868 (1.70%)
5 errors : 119 (0.23%)
6 errors : 13 (0.03%)
7 errors : 1 (0.00%)


In [4]:
import pandas as pd

CLEAN_PATH = "/kaggle/input/datasets/shruteesaha/49k-clean-dataset/49k_clean_sentences.csv"
NOISY_PATH = "/kaggle/input/datasets/shruteesaha/51k-after-final-injection/51k_after_dictionary_spelling_injection (1).csv"

clean_df = pd.read_csv(CLEAN_PATH)
noisy_df = pd.read_csv(NOISY_PATH)

clean_test = clean_df.sample(
    n=4900,
    random_state=42
)

noisy_test = noisy_df.sample(
    n=5100,
    random_state=42
)

clean_train = clean_df.drop(clean_test.index)
noisy_train = noisy_df.drop(noisy_test.index)

test_df = pd.concat(
    [clean_test, noisy_test],
    ignore_index=True
).sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

train_df = pd.concat(
    [clean_train, noisy_train],
    ignore_index=True
).sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

clean_test.to_csv("clean_test_4900.csv", index=False)
noisy_test.to_csv("noisy_test_5100.csv", index=False)

test_df.to_csv("test_dataset_10000.csv", index=False)
train_df.to_csv("train_dataset_90000.csv", index=False)

print("Clean Train :", len(clean_train))
print("Noisy Train :", len(noisy_train))

print("Clean Test  :", len(clean_test))
print("Noisy Test  :", len(noisy_test))

print("Train Total :", len(train_df))
print("Test Total  :", len(test_df))

Clean Train : 44100
Noisy Train : 45900
Clean Test  : 4900
Noisy Test  : 5100
Train Total : 90000
Test Total  : 10000


In [6]:
import pandas as pd

train_df = pd.read_csv("train_dataset_90000.csv")

train_df = train_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

train_df.to_csv(
    "train_dataset_90000_shuffled.csv",
    index=False
)

print(train_df.head(50))
print("Total rows:", len(train_df))

                                             sentence error_label
0   ডিএনএ সূত্ৰত এটা নিউক্লিয়টাইডৰ শৰ্কৰাৰ লগত আন...           1
1   বৰ্তমান টাই খাময়াংসকল ঘাইকৈ অসমক শিৱসাগৰ, যোৰ...       2,7,8
2   তেওঁলোকৰ ফাণ্টিনা (Fantina),বেল্লেলা (Bellela)...           1
3   এক নিৰ্দিষ্ট সময়ৰ মূৰে মূৰে কিমান ৰছী খোল খাই...           0
4   তেওঁ তিনিটা অনুপ্ৰেৰণামূলক ৱেব ছাইটৰো সহ-সংগঠক...           0
5   তাৰপিছত তেখেত ভাৰতলৈ উভতি আহে আৰু ৰামানুজন গণি...           0
6   ইয়াৰ পৰাই তেওঁ ৰাজনীতিৰ প্ৰাৰম্ভিক শিক্ষা গ্ৰ...    2,9,1,1D
7   সংবাদভিত্তিক সমালোচনা বহুলকৈ বাতৰি-কাকত বা অন্...           0
8   তেওঁ আব্দুছৰ দ্বাৰা অংকিত সৰল ৰেখাডাল চাই কৈছি...           1
9   খোৰোং বা ধোন্দৰ মুখৰ ব্যাস সাধাৰণতে তিনি ইঞ্চি...           0
10  পানী গৰম কৰিব পৰা তথা সকলো স্থানতে এক জটিল নেট...           0
11  গ্ৰেপ্তাৰৰ পাছত তেওঁক কুমিল্লাৰ পৰা কলকাতালৈ ল...           0
12  অপুৰ সংসাৰ নিৰ্মাণৰ পাছত সত্যজিৎে দেৱী ছবিৰ কা...           0
13  কিন্তু কম্পিউটাৰ আবিষ্কাৰে সাংখ্যিক বিশ্লেষণৰ ...           8
14  প্ৰথমা

In [7]:
test_input = "/kaggle/working/test_dataset_10000.csv"
test_df = pd.read_csv(test_input)
print(test_df.head(50).to_string(index=False))

                                                                                                                                                                                  sentence error_label
                                                                           কেতিয়াবা পৃথিৱীৰ কোনো বস্তুৰ স্থিতি অথবা পৰিবৰ্তন বুজাবলৈও ইয়াক বৱহাৰ কৰিবা হয়, যেনে বতৰ, পৃথিৱীৰ ভূগোল আদি।         3,1
                                                                       এইজন ডাৰউইনৰ জীৱন চিন্তা আৰু কৰ্মৰ বিৱৰণ দি লিখা প্ৰথম আৰু একমাত্ৰ অসমীয়া কিতাপখনেই হোমেন বৰগোহাঞিয়ে লিখা ডাৰউইন।           0
                                                                           তাৰ পাছত তেওঁ মিনিম’ট’ প্ৰতিযোগিতা সমূহত ভাগ ল’বলৈ আৰম্ভ কৰে, ১৯৯১ৰ ভিতৰত তেওঁ কেইবাটাও স্থানীয় খিটাপ জয় কৰে।           0
                                                                                         তেওঁ নিজৰ নিকট সহযোগীসকলে আঘাত পাই বুলি তেওঁলোকৰ পৰামৰ্শ অস্বীকাৰ বা অবিশ্বাস কৰিবলৈ নিবিচাৰিছিল।           0
     